# Identity Resolution: Goodreads, Amazon, DBpedia (Phase II)

**Entity = edition.** Two records match if they describe the same edition of a book
(same publication, format and ISBN), not just the same work. Different editions of the
same novel ("Where the Crawdads Sing" hardcover vs. large print) are *non-matches*.

Workflow, following the PyDI exercise (WDI05):
1. Load the three translated XML files and add cleaned helper columns
2. Gold standard: train / validation / test per dataset pair
3. Blocking: combine author-surname and title keys, evaluate pair completeness and reduction ratio
4. Features: PyDI comparators + "both values present" indicators
5. Matchers: simple rule-based baseline and ML models (selected on validation)
6. Apply the best model to all candidates, enforce one-to-one matching, evaluate on the test set
7. Save correspondences for data fusion

The ISBN is **not** used anywhere in matching; it was only used to build the gold standard.

## 1. Setup

In [1]:
import logging
import os
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display

os.makedirs("logs", exist_ok=True)
logging.basicConfig(level=logging.INFO, format="[%(levelname)-5s] %(name)s - %(message)s",
                    handlers=[logging.FileHandler("logs/identity_resolution.log", encoding="utf-8")], force=True)

from PyDI.io import load_xml
from PyDI.entitymatching import FeatureExtractor, MLBasedMatcher, RuleBasedMatcher, GreedyOneToOneMatchingAlgorithm
from PyDI.entitymatching.base import BaseComparator
from PyDI.entitymatching.comparators import StringComparator, NumericComparator
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator

DATA_DIR = Path("data")
GS_DIR = Path("gold_standard")
OUT_DIR = DATA_DIR / "output" / "identity_resolution"
CORR_DIR = DATA_DIR / "correspondences"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = [("goodreads", "amazon"), ("goodreads", "dbpedia"), ("amazon", "dbpedia")]
SEED = 42

c:\Users\safas\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load data and add cleaned helper columns

The matchers compare the original attributes plus a few cleaned versions:
* `title_clean`: lower case, without bracket text such as "(Movie Tie-In)" or DBpedia's "(novel)",
  without subtitle and leading article
* `title_after_colon`: the part after a series name, e.g. "The Lost Fleet: Dauntless" -> "dauntless"
* `first_author`, `author_surname`: handles "(Goodreads Author)", "Vance, J. D." and "Jr."
* numeric `year`, `page_count`, `rating`, `ratings_count`, `price`

In [2]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "phd", "md", "dr", "ms", "mr", "mrs"}


def ascii_lower(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower() if isinstance(s, str) else ""


def clean_title(t):
    t = ascii_lower(t)
    t = re.sub(r"^(.+?)\s{2,}.*\s{2,}\d+$", r"\1", t)   # DBpedia infobox titles "A  B  1" -> keep only "A"
    t = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", t)
    t = re.sub(r"\s*(:| - ).*$", "", t)
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return re.sub(r"^(the|a|an) ", "", t)


def first_author(a):
    if isinstance(a, list):
        a = a[0] if a else ""
    return re.sub(r"\(.*?\)", "", a).strip() if isinstance(a, str) else ""


def surname(a):
    a = re.sub(r"\(.*?\)", "", a or "").strip()
    if "," in a:
        w = re.sub(r"[^a-z ]", " ", ascii_lower(a.split(",")[0])).split()
        return w[-1] if w else ""
    w = [x for x in re.sub(r"[^a-z ]", " ", ascii_lower(a)).split() if x not in SUFFIXES]
    return w[-1] if w else ""


PUB_STOP = {"books", "book", "publishing", "publishers", "publisher", "publications", "publication",
            "press", "inc", "ltd", "llc", "co", "company", "and", "the", "group", "limited",
            "corp", "corporation"}
PUB_ABBR = {"pub": "publishing", "pubs": "publishing", "intl": "international", "univ": "university"}


def normalize_publisher(s):
    if not isinstance(s, str) or not s.strip():
        return ""
    s = ascii_lower(s)
    s = re.sub(r"\([^)]*\)", " ", s)      # remove "(publisher)", "(US)" ...
    s = re.sub(r"[^a-z0-9 ]", " ", s)     # remove punctuation
    words = [PUB_ABBR.get(w, w) for w in s.split()]
    return " ".join(w for w in words if w not in PUB_STOP)


def load(name):
    df = load_xml(DATA_DIR / f"{name}.xml", name=name)
    df.attrs["source"] = name
    for col in ["title", "authors_author", "publisher", "publication_date", "page_count", "language",
                "format", "edition", "rating", "ratings_count", "price"]:
        if col not in df.columns:
            df[col] = None
    df["title_clean"] = df["title"].map(clean_title)
    df["title_after_colon"] = df["title"].map(
        lambda t: clean_title(t.split(":", 1)[1]) if isinstance(t, str) and ":" in t else clean_title(t))
    df["first_author"] = df["authors_author"].map(first_author)
    df["author_surname"] = df["first_author"].map(surname)
    df["publisher_clean"] = df["publisher"].map(ascii_lower)
    df["publisher_norm"] = df["publisher"].map(normalize_publisher)   # normalised publisher for the ML features
    df["format_clean"] = df["format"].map(ascii_lower)
    df["language_clean"] = df["language"].map(ascii_lower)
    df["year"] = pd.to_numeric(df["publication_date"].astype("string").str[:4], errors="coerce")
    for c in ["page_count", "rating", "ratings_count", "price"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


data = {name: load(name) for name in ["goodreads", "amazon", "dbpedia"]}
for name, df in data.items():
    print(f"{name:<10} {len(df):>7,} records")
display(data["amazon"][["id", "title", "title_clean", "first_author", "author_surname", "publisher", "year", "page_count", "format"]].head())

goodreads   52,478 records
amazon      40,000 records
dbpedia     59,270 records


,id,title,title_clean,first_author,author_surname,publisher,year,page_count,format
0,amazon_id_1,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Penguin Audio,2019,NaN,NaN
1,amazon_id_2,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Corsair,2019,NaN,Hardcover
2,amazon_id_3,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Random House Large Print,2018,496.0,Paperback
3,amazon_id_4,Where the Crawdads Sing (Movie Tie-In),where the crawdads sing,Delia Owens,owens,G.P. Putnam's Sons,2022,400.0,Paperback
4,amazon_id_5,Where the Crawdads Sing,where the crawdads sing,Delia Owens,owens,Corsair,2019,NaN,Paperback


## 3. Gold standard: train / validation / test

`gold_standard/<pair>_train.csv` and `<pair>_test.csv` come from `build_gold_standard.py`
(ISBN-based, 539 pairs checked by hand, wrong pairs removed). We split 25% of the training
pairs off as a **validation set** (stratified, fixed seed) for choosing blocking, models and
thresholds. The **test set is only used once, for the final evaluation.**

In [3]:
from sklearn.model_selection import train_test_split


def read_gs(path):
    df = pd.read_csv(path)
    df["label"] = df["label"].astype(str).str.upper().eq("TRUE")
    return df[["id1", "id2", "label"]]


gold = {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    train_full = read_gs(GS_DIR / f"{p}_train.csv")
    train, val = train_test_split(train_full, test_size=0.25, stratify=train_full["label"], random_state=SEED)
    gold[p] = {"train": train.reset_index(drop=True), "validation": val.reset_index(drop=True),
               "test": read_gs(GS_DIR / f"{p}_test.csv")}
    print(p, {k: f"{len(v)} ({v.label.sum()} TRUE)" for k, v in gold[p].items()})

goodreads_amazon {'train': '624 (201 TRUE)', 'validation': '209 (67 TRUE)', 'test': '357 (97 TRUE)'}
goodreads_dbpedia {'train': '633 (209 TRUE)', 'validation': '212 (70 TRUE)', 'test': '363 (108 TRUE)'}
amazon_dbpedia {'train': '597 (171 TRUE)', 'validation': '199 (57 TRUE)', 'test': '342 (89 TRUE)'}


## 4. Blocking

Comparing all pairs would mean 2–3 billion comparisons per dataset pair. We tested three
blocking keys on the training + validation pairs (target: pair completeness above 97%):

* **author surname** alone misses co-authors listed in a different order and pseudonyms
  (Richard Bachman / Stephen King, J.D. Robb / Nora Roberts)
* **cleaned title** alone misses Amazon/DBpedia titles with series prefixes or subtitles
* the **union** of surname, cleaned title and title-after-colon keys keeps almost all matches

Reduction ratio = share of all possible pairs that are *not* compared.

In [4]:
def key_pairs(A, B, col_a, col_b):
    a = A.loc[A[col_a] != "", ["id", col_a]].rename(columns={col_a: "k"})
    b = B.loc[B[col_b] != "", ["id", col_b]].rename(columns={col_b: "k"})
    return a.merge(b, on="k", suffixes=("1", "2"))[["id1", "id2"]]


BLOCKING_KEYS = {
    "surname": [("author_surname", "author_surname")],
    "title": [("title_clean", "title_clean")],
    "surname OR title (+ after colon)": [("author_surname", "author_surname"), ("title_clean", "title_clean"),
                                         ("title_clean", "title_after_colon"), ("title_after_colon", "title_clean")],
}


def block(A, B, keys):
    return pd.concat([key_pairs(A, B, ca, cb) for ca, cb in keys]).drop_duplicates().reset_index(drop=True)


def blocking_metrics(cands, gs_pairs, n_all):
    pos = set(zip(gs_pairs.loc[gs_pairs.label, "id1"], gs_pairs.loc[gs_pairs.label, "id2"]))
    found = len(pos & set(zip(cands.id1, cands.id2)))
    return {"candidates": len(cands), "pair completeness": found / len(pos), "reduction ratio": 1 - len(cands) / n_all}


rows, candidates = [], {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    A, B = data[a], data[b]
    dev = pd.concat([gold[p]["train"], gold[p]["validation"]])
    for name, keys in BLOCKING_KEYS.items():
        t = time.time()
        c = block(A, B, keys)
        m = blocking_metrics(c, dev, len(A) * len(B))
        rows.append({"pair": p, "blocking": name, **m, "seconds": round(time.time() - t, 1)})
        if name.startswith("surname OR"):
            candidates[p] = c
            if os.environ.get("QUICK"):   # debugging only: small sample of candidates
                candidates[p] = c.sample(min(len(c), 30_000), random_state=SEED)
blocking_results = pd.DataFrame(rows)
blocking_results.to_csv(OUT_DIR / "blocking_results.csv", index=False)
display(blocking_results.style.format({"candidates": "{:,}", "pair completeness": "{:.1%}", "reduction ratio": "{:.4%}"}))

,pair,blocking,candidates,pair completeness,reduction ratio,seconds
0,goodreads_amazon,surname,"1,299,848",96.6%,99.9381%,2.200000
1,goodreads_amazon,title,"36,272",77.6%,99.9983%,0.200000
2,goodreads_amazon,surname OR title (+ after colon),"1,320,641",100.0%,99.9371%,2.400000
3,goodreads_dbpedia,surname,"1,062,280",92.1%,99.9658%,1.900000
4,goodreads_dbpedia,title,"21,705",61.3%,99.9993%,0.200000
5,goodreads_dbpedia,surname OR title (+ after colon),"1,074,609",98.2%,99.9655%,2.100000
6,amazon_dbpedia,surname,"1,200,724",92.1%,99.9494%,1.500000
7,amazon_dbpedia,title,"29,607",76.3%,99.9988%,0.100000
8,amazon_dbpedia,surname OR title (+ after colon),"1,217,067",97.4%,99.9487%,1.900000


## 5. Features

PyDI comparators on title, author, publisher, year, page count, format and language.
Missing values get similarity 0 in PyDI, which looks the same as "completely different".
Because DBpedia often lacks page count and format, we add **presence indicators**
(1 if both records have a value), so the models can tell "different" from "unknown".

In [5]:
class BothPresent(BaseComparator):
    """1.0 if both records have a value in `column`, else 0.0."""

    def __init__(self, column):
        super().__init__(name=f"BothPresent({column})")
        self.column = column

    def compare(self, record1, record2):
        def ok(v):
            if isinstance(v, (list, tuple)):
                return len(v) > 0
            try:
                if v is None or pd.isna(v):
                    return False
            except (TypeError, ValueError):
                pass
            return str(v).strip() not in ("", "nan", "<NA>")
        return float(ok(record1.get(self.column)) and ok(record2.get(self.column)))

comparators = [
    StringComparator("title_clean", similarity_function="jaro_winkler"),
    StringComparator("title_clean", similarity_function="levenshtein"),
    StringComparator("title", similarity_function="jaccard", tokenization="word", preprocess=str.lower),
    StringComparator("title_after_colon", similarity_function="jaro_winkler"),
    StringComparator("first_author", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("authors_author", similarity_function="jaccard", tokenization="word",
                     preprocess=str.lower, list_strategy="best_match"),
    StringComparator("publisher_norm", similarity_function="jaccard", tokenization="word"),
    NumericComparator("year", method="absolute_difference", max_difference=10),
    NumericComparator("page_count", method="relative_difference", max_difference=0.5),
    StringComparator("format_clean", similarity_function="jaro_winkler"),
    StringComparator("language_clean", similarity_function="jaro_winkler"),
    BothPresent("year"), BothPresent("page_count"), BothPresent("publisher"), BothPresent("format"),
]
feature_extractor = FeatureExtractor(comparators)

features = {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    features[p] = {split: feature_extractor.create_features(data[a], data[b], gs[["id1", "id2"]],
                                                            labels=gs["label"], id_column="id")
                   for split, gs in gold[p].items()}
FEATURE_COLUMNS = [c for c in features[f"{PAIRS[0][0]}_{PAIRS[0][1]}"]["train"].columns if c not in ("id1", "id2", "label")]
print(len(FEATURE_COLUMNS), "features")
display(features["goodreads_amazon"]["train"].head(3).T)

15 features


,0,1,2
id1,goodreads_id_494,goodreads_id_784,goodreads_id_41047
id2,amazon_id_9643,amazon_id_13424,amazon_id_25877
"StringComparator(title_clean, jaro_winkler, tokenization=char, list_strategy=None)",1.0,1.0,0.548622
"StringComparator(title_clean, levenshtein, tokenization=char, list_strategy=None)",1.0,1.0,0.105263
"StringComparator(title, jaccard, tokenization=word, list_strategy=None)",1.0,1.0,0.0
"StringComparator(title_after_colon, jaro_winkler, tokenization=char, list_strategy=None)",1.0,1.0,0.412698
"StringComparator(first_author, jaro_winkler, tokenization=char, list_strategy=None)",1.0,0.966667,0.545752
"StringComparator(authors_author, jaccard, tokenization=word, list_strategy=best_match)",0.5,0.666667,0.25
"StringComparator(publisher_norm, jaccard, tokenization=word, list_strategy=None)",0.0,1.0,0.0
"NumericComparator(year, absolute_difference, list_strategy=None)",0.9,0.0,0.0


In [6]:
import pandas as pd
r1, r2 = data["goodreads"].iloc[0], data["amazon"].iloc[0]
v = r1.get("year")
print("type:", type(v), "| value:", repr(v))
print("pd.isna:", pd.isna(v))
print("pandas version:", pd.__version__)
print("BothPresent:", BothPresent("year").compare(r1, r2))

type: <class 'numpy.int64'> | value: np.int64(2008)
pd.isna: False
pandas version: 2.3.3
BothPresent: 1.0


## 6. Matchers

### 6.1 Rule-based baseline
A simple weighted rule (title 0.5, author 0.3, year 0.2), with the threshold chosen on the
validation set. This is only a **baseline** to compare the ML models against; the tuned
rule-based matcher is developed separately.

In [7]:
from sklearn.metrics import f1_score, precision_score, recall_score

BASELINE = {"StringComparator(title_clean, jaro_winkler": 0.5, "StringComparator(first_author, jaro_winkler": 0.3,
            "NumericComparator(year, absolute_difference": 0.2}


def baseline_score(df):
    cols = {prefix: next(c for c in FEATURE_COLUMNS if c.startswith(prefix)) for prefix in BASELINE}
    return sum(df[cols[k]] * w for k, w in BASELINE.items())


def scores(y, pred):
    return {"precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0)}


baseline = {}
for p in features:
    val = features[p]["validation"]
    best = max(((t, f1_score(val.label, baseline_score(val) >= t)) for t in [x / 100 for x in range(50, 100)]),
               key=lambda x: x[1])
    baseline[p] = best[0]
    print(f"{p:<20} best threshold {best[0]:.2f} -> validation F1 {best[1]:.3f}")

goodreads_amazon     best threshold 0.99 -> validation F1 0.698
goodreads_dbpedia    best threshold 0.65 -> validation F1 0.611
amazon_dbpedia       best threshold 0.99 -> validation F1 0.596


### 6.1b Linearly Weighted Matching Rule
#### Step 1: Where does the baseline fail?

In [8]:
COLS = ["title", "first_author", "publisher", "publication_date", "format", "page_count"]

baseline_errors = {}
for p in features:
    a, b = p.split("_")
    A, B = data[a].set_index("id"), data[b].set_index("id")
    va = features[p]["validation"].copy()
    va["pred"] = baseline_score(va) >= baseline[p]

    rows = []
    for kind, mask in [("FP", va.pred & ~va.label), ("FN", ~va.pred & va.label)]:
        for x, y in zip(va.loc[mask, "id1"], va.loc[mask, "id2"]):
            r1, r2 = A.loc[x], B.loc[y]
            rows.append({"error": kind,
                         **{f"{c} 1": r1[c] for c in COLS}, **{f"{c} 2": r2[c] for c in COLS}})
    baseline_errors[p] = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 35)
for p, err in baseline_errors.items():
    print(f"\n=== {p}: first 8 false positives ===")
    display(err[err.error == "FP"].head(8))
    print(f"=== {p}: first 8 false negatives ===")
    display(err[err.error == "FN"].head(8))


=== goodreads_amazon: first 8 false positives ===


,error,title 1,first_author 1,publisher 1,publication_date 1,format 1,page_count 1,title 2,first_author 2,publisher 2,publication_date 2,format 2,page_count 2
0,FP,Hotel on the Corner of Bitter a...,Jamie Ford,Ballantine Books,2009-01-27,Hardcover,290.0,Hotel on the Corner of Bitter a...,Jamie Ford,Ballentine Books,2009-10-06,Paperback,NaN
1,FP,The Innocent,David Baldacci,Grand Central Publishing,2012-04-17,Hardcover,422.0,The Innocent (Will Robie Series...,David Baldacci,Grand Central Publishing,2012-10-09,Audio CD,NaN
2,FP,Face the Fire,Nora Roberts,Judy Piatkus,2002-05-01,Paperback,358.0,Face The Fire - Large Print Edi...,Nora Roberts,Jove Books Bookspan Large Prin,2002-01-01,NaN,560.0
3,FP,Past Tense,Lee Child,Delacorte Press,2018-11-05,Hardcover,382.0,Past Tense: (Jack Reacher 23),Lee Child,RANDOM HOUSE UK,2018-11-01,Paperback,NaN
4,FP,Simple Genius,David Baldacci,Warner Books,2007-04-24,Hardcover,420.0,Simple Genius,David Baldacci,Hachette Audio,2007-04-24,Audio CD,NaN
5,FP,The Escape,David Baldacci,Grand Central Publishing,2014-11-18,Hardcover,470.0,The Escape (John Puller Series),David Baldacci,Grand Central Publishing,2014-11-18,Audio CD,NaN
6,FP,The Girl with the Make-Believe ...,Julia Quinn,Avon,2017-05-30,ebook,384.0,The Girl with the Make-Believe ...,Julia Quinn,HarperAudio,2017-05-30,Audio CD,NaN
7,FP,House Rules,Jodi Picoult,Atria Books,2010-03-02,Hardcover,532.0,House Rules: A Novel,Jodi Picoult,Washington Square Press,2010-11-09,Paperback,560.0


=== goodreads_amazon: first 8 false negatives ===


,error,title 1,first_author 1,publisher 1,publication_date 1,format 1,page_count 1,title 2,first_author 2,publisher 2,publication_date 2,format 2,page_count 2
15,FN,The Boys in the Boat: Nine Amer...,Daniel James Brown,Viking,2013-06-04,Hardcover,404.0,The Boys in the Boat: Nine Amer...,Daniel Brown,Viking,2013-01-01,Hardcover,416.0
16,FN,The Giving Tree,Shel Silverstein,HarperCollins Publishers,NaN,Hardcover,64.0,The Giving Tree,Shel Silverstein,HarperCollins,1964-01-01,Hardcover,57.0
17,FN,Savage Run,C.J. Box,G.P. Putnam's Sons,2003-05-06,Paperback,304.0,Savage Run (A Joe Pickett Novel),C. J. Box,Prime Crime,2003-01-01,Paperback,NaN
18,FN,What If? Serious Scientific Ans...,Randall Munroe,Houghton Mifflin Harcourt,2014-09-02,Hardcover,303.0,What If?: Serious Scientific An...,Randall Munroe,Dey Street Books,2014-09-02,Hardcover,320.0
19,FN,Sovereign,C.J. Sansom,Viking Books,2007-04-01,Hardcover,583.0,Sovereign (Matthew Shardlake My...,C. J. Sansom,Viking Adult,2007-03-22,Hardcover,592.0
20,FN,Excalibur,Bernard Cornwell,St. Martin's Griffin,1999-07-16,Paperback,448.0,Excalibur (The Warlord Chronicl...,Bernard Cornwell,St. Martin's Griffin Press,1999-07-16,Paperback,436.0
21,FN,"Eat, Pray, Love",Elizabeth Gilbert,Riverhead Books,NaN,Paperback,368.0,"Eat, Pray, Love: One Woman's Se...",Elizabeth Gilbert,Riverhead Books,2007-01-30,Paperback,400.0
22,FN,The Lovely Bones,Alice Sebold,"Little, Brown and Company",NaN,Mass Market Paperback,372.0,The Lovely Bones,Alice Sebold,"Little, Brown and Company",2006-09-01,Paperback,384.0



=== goodreads_dbpedia: first 8 false positives ===


,error,title 1,first_author 1,publisher 1,publication_date 1,format 1,page_count 1,title 2,first_author 2,publisher 2,publication_date 2,format 2,page_count 2
0,FP,Tales of Known Space: The Unive...,Larry Niven,Del Rey,1981-03-12,Paperback,240.0,Tales of Known Space,Larry Niven,Ballantine Books,NaN,None,240.0
1,FP,Leave Myself Behind,Bart Yates,Kensington,2004-04-01,Paperback,256.0,Leave Myself Behind,Bart Yates,Kensington Publishing,2003-03-04,None,NaN
2,FP,Not Quite What I Was Planning: ...,Larry Smith,Harper Perennial,2008-02-05,Paperback,225.0,The Triumph of the Sun,Wilbur Smith,Macmillan Publishers,2005-01-01,None,NaN
3,FP,Term Limits,Vince Flynn,Pocket Star Books,1999-07-01,Paperback,612.0,Term Limits (novel),Vince Flynn,"Cloak and Dagger, Incorporated",NaN,None,403.0
4,FP,Theft of Swords,Michael J. Sullivan,Orbit,2011-11-23,Paperback,649.0,Dear Blue Sky,Mary Sullivan,Puffin Books,2012-01-01,None,248.0
5,FP,Bunny,Mona Awad,Head of Zeus,2019-06-13,Paperback,373.0,Bunny (novel),Mona Awad,Vikings,2019-06-11,None,305.0
6,FP,Alice in Sunderland,Bryan Talbot,Dark Horse Books,2007-05-01,Hardcover,328.0,Alice in Sunderland,Bryan Talbot,NaN,2007-04-05,None,328.0
7,FP,Islandia,Austin Tappan Wright,Overlook Press,2006-07-25,Paperback,1040.0,Islandia (novel),Austin Tappan Wright,Farrar & Rinehart,1942-01-01,None,1014.0


... (2 outputs left out)


=== amazon_dbpedia: first 8 false positives ===


,error,title 1,first_author 1,publisher 1,publication_date 1,format 1,page_count 1,title 2,first_author 2,publisher 2,publication_date 2,format 2,page_count 2
0,FP,The Hanging Tree: The Sixth Riv...,Ben Aaronovitch,Gollancz,2016-12-08,Audio CD,NaN,The Hanging Tree (Aaronovitch n...,Ben Aaronovitch,Victor Gollancz Ltd,2016-11-03,None,NaN
1,FP,Kiss the Girls,James Patterson,Little Brown,1995-01-01,NaN,NaN,Kiss the Girls (novel),James Patterson,"Little, Brown and Company",1995-01-11,None,464.0
2,FP,A Painted House: A Novel (Rando...,John Grisham,Random House,2001-02-06,Hardcover,562.0,A Painted House,John Grisham,Doubleday (publisher),2001-02-06,None,388.0
3,FP,Shade: A Tale of Two Presidents,Pete Souza,"Little, Brown and Company",2018-10-16,Hardcover,240.0,Shade: A Tale of Two Presidents,Pete Souza,"Little, Brown and Company",2018-01-01,None,NaN
4,FP,"Eragon (Inheritance, Book 1)",Christopher Paolini,Listening Library,2003-08-26,NaN,NaN,Eragon,Christopher Paolini,Alfred A. Knopf,2003-08-26,None,509.0
5,FP,The School for Good Mothers,Jessamine Chan,Simon & Schuster,2022-01-04,Paperback,NaN,The School for Good Mothers,Jessamine Chan,Simon & Schuster,2022-01-04,None,336.0
6,FP,Billy Summers,Stephen King,Hodder & Stoughton,2021-01-01,Paperback,NaN,Billy Summers,Stephen King,Charles Scribner's Sons,2021-08-03,None,528.0
7,FP,The Shadow Of What Was Lost,James Islington,Aslaradis Publishing,2014-08-01,Paperback,604.0,The Shadow of What Was Lost,James Islington,Orbit Books,2014-01-01,None,725.0


=== amazon_dbpedia: first 8 false negatives ===


,error,title 1,first_author 1,publisher 1,publication_date 1,format 1,page_count 1,title 2,first_author 2,publisher 2,publication_date 2,format 2,page_count 2
23,FN,The Namesake,Jhumpa Lahiri,NaN,NaN,NaN,NaN,The Namesake (novel),Jhumpa Lahiri,NaN,NaN,None,291.0
24,FN,Alcoholics Anonymous: The Story...,Alcoholics Anonymous World Serv...,Alcoholics Anonymous World Serv...,2003-01-01,Hardcover,575.0,The Big Book (Alcoholics Anonym...,Bill W.,Alcoholics Anonymous World Serv...,1955-01-01,None,NaN
25,FN,Crucial Conversations Tools for...,Joseph Grenny,McGraw Hill,2011-09-09,Paperback,272.0,Crucial Conversations: Tools fo...,Al Switzler,GradeGuru,2002-01-01,None,288.0
26,FN,Kitchen Confidential,Anthony Bourdain,Bloomsbury Pub Ltd,2000-01-01,Hardcover,320.0,Kitchen Confidential,Anthony Bourdain,Bloomsbury Publishing,NaN,None,320.0
27,FN,Magnus Chase and the Gods of As...,Rick Riordan,Disney Hyperion,2016-10-04,Hardcover,480.0,The Hammer of Thor,Rick Riordan,Disney Press,2016-10-04,None,528.0
28,FN,Consent to Kill: A Thriller (8)...,Vince Flynn,Pocket Star,2006-09-01,Mass Market Paperback,704.0,Consent to Kill,Vince Flynn,Simon & Schuster,2005-10-11,None,NaN
29,FN,La Sombra del Viento (Spanish E...,Carlos Ruiz Zafón,Planeta,2006-03-21,Paperback,567.0,The Shadow of the Wind,Carlos Ruiz Zafón,Grupo Planeta,2001-01-01,None,565.0
30,FN,Finding Me: A Decade of Darknes...,Michelle Burford,Hachette Books,2015-03-10,Paperback,280.0,Finding Me: A Decade of Darknes...,Michelle Knight,Weinstein Books,2014-05-06,None,280.0


In [9]:
import numpy as np


def fcol(prefix):
    return next(c for c in FEATURE_COLUMNS if c.startswith(prefix))


TITLE = fcol("StringComparator(title_clean, jaro_winkler")
AUTHOR = fcol("StringComparator(first_author, jaro_winkler")
YEAR = fcol("NumericComparator(year")
PUB = fcol("StringComparator(publisher_clean")
PAGES = fcol("NumericComparator(page_count")
FMT = fcol("StringComparator(format_clean")


def edition_missing(r):
    return min(r["BothPresent(publisher)"], r["BothPresent(page_count)"], r["BothPresent(format)"]) == 0


FP_FACTORS = {
    "title_clean similar (>= 0.9)": lambda r: r[TITLE] >= 0.9,
    "first_author similar (>= 0.9)": lambda r: r[AUTHOR] >= 0.9,
    "year similar (>= 0.9)": lambda r: r["BothPresent(year)"] == 1 and r[YEAR] >= 0.9,
    "publisher_clean differs (< 0.5)": lambda r: r["BothPresent(publisher)"] == 1 and r[PUB] < 0.5,
    "page_count differs (< 0.9)": lambda r: r["BothPresent(page_count)"] == 1 and r[PAGES] < 0.9,
    "format_clean differs (< 0.9)": lambda r: r["BothPresent(format)"] == 1 and r[FMT] < 0.9,
    "publisher / page_count / format missing": edition_missing,
}
FN_FACTORS = {
    "title_clean differs (< 0.9)": lambda r: r[TITLE] < 0.9,
    "first_author differs (< 0.9)": lambda r: r[AUTHOR] < 0.9,
    "year missing": lambda r: r["BothPresent(year)"] == 0,
    "year differs (< 0.9)": lambda r: r["BothPresent(year)"] == 1 and r[YEAR] < 0.9,
    "title_clean, first_author, year similar, below threshold":
        lambda r: r[TITLE] >= 0.9 and r[AUTHOR] >= 0.9 and r["BothPresent(year)"] == 1 and r[YEAR] >= 0.9,
    "publisher / page_count / format missing": edition_missing,
}

table = {}
for p in features:
    va = features[p]["validation"].copy()
    pred = baseline_score(va) >= baseline[p]
    fp, fn = va[pred & ~va.label], va[~pred & va.label]
    col = {("", "validation pairs"): len(va), ("FP", "total"): len(fp), ("FN", "total"): len(fn)}
    for name, f in FP_FACTORS.items():
        col[("FP", name)] = sum(f(r) for _, r in fp.iterrows())
    for name, f in FN_FACTORS.items():
        col[("FN", name)] = sum(f(r) for _, r in fn.iterrows())
    table[p] = col

factors = pd.DataFrame(table)


def with_pct(col):
    out = col.astype(object)
    for (group, name), n in col.items():
        if group in ("FP", "FN") and name != "total":
            total = col[(group, "total")]
            out[(group, name)] = f"{int(n)} ({n / total:.0%})" if total else "0"
    return out


display(factors.apply(with_pct))

StopIteration: 

In [ ]:
features["goodreads_amazon"]["validation"]["BothPresent(year)"].value_counts()

BothPresent(year)
1.0    189
0.0     20
Name: count, dtype: int64

#### Step 2: Tuned linear weighted rule

#### Publisher normalization

- **Problem:** 62–100% of the false negatives differ in `publisher_clean` ("Little, Brown" vs. "Little, Brown and Company" → 0.20).
- **Solution:** `publisher_norm` removes punctuation, brackets and generic words ("inc", "ltd", "press") → "little brown" = 1.0.
- **Kept** only if the validation F1 improves for all three pairs.

In [ ]:
import re

PUB_STOP = {"books", "book", "publishing", "publishers", "publisher", "publications", "publication",
            "press", "inc", "ltd", "llc", "co", "company", "and", "the", "group", "limited",
            "corp", "corporation"}
PUB_ABBR = {"pub": "publishing", "pubs": "publishing", "intl": "international", "univ": "university"}


def normalize_publisher(s):
    if not isinstance(s, str) or not s.strip():
        return ""
    s = ascii_lower(s)
    s = re.sub(r"\([^)]*\)", " ", s)      # remove "(publisher)", "(US)" ...
    s = re.sub(r"[^a-z0-9 ]", " ", s)     # remove punctuation
    words = [PUB_ABBR.get(w, w) for w in s.split()]
    return " ".join(w for w in words if w not in PUB_STOP)


def word_jaccard(x, y):
    a, b = set(x.split()), set(y.split())
    return len(a & b) / len(a | b) if a and b else 0.0


for name, df in data.items():
    df["publisher_norm"] = df["publisher"].map(normalize_publisher)

PUB_NORM = "Jaccard(publisher_norm)"


def add_pub_norm(df, a, b):
    """Adds the column PUB_NORM: word Jaccard of publisher_norm for each pair (id1, id2)."""
    m1 = data[a].set_index("id")["publisher_norm"]
    m2 = data[b].set_index("id")["publisher_norm"]
    df[PUB_NORM] = [word_jaccard(x, y) for x, y in zip(df["id1"].map(m1), df["id2"].map(m2))]
    return df


for p in features:
    a, b = p.split("_")
    for split in features[p]:
        add_pub_norm(features[p][split], a, b)

# check a few examples
for name, df in data.items():
    print(name)
    display(df[["publisher", "publisher_norm"]].dropna().drop_duplicates().head(8))

goodreads


,publisher,publisher_norm
0,Scholastic Press,scholastic
1,Scholastic Inc.,scholastic
2,Harper Perennial Modern Classics,harper perennial modern classics
3,Modern Library,modern library
4,"Little, Brown and Company",little brown
5,Alfred A. Knopf,alfred a knopf
6,Signet Classics,signet classics
7,HarperCollins,harpercollins


amazon


,publisher,publisher_norm
0,Penguin Audio,penguin audio
1,Corsair,corsair
2,Random House Large Print,random house large print
3,G.P. Putnam's Sons,g p putnam s sons
5,Penguin Publishing Group,penguin
10,Thorndike Press,thorndike
11,BLACK SWAN,black swan
12,Riverhead Books,riverhead


dbpedia


,publisher,publisher_norm
0,Berkley Books,berkley
1,Henry Holt and Company,henry holt
3,University of Texas Press,university of texas
8,Time (magazine),time
16,Executive Committee of the Communist Internati...,executive committee of communist international
19,"William Heinemann Ltd, London",william heinemann london
22,G. P. Putnam's Sons,g p putnam s sons
28,University of Virginia Press,university of virginia


In [ ]:
# ============================================================
# 1) TRAIN: learn the weights of 4 methods, all with publisher_norm
#    Uses ONLY the gold-standard training set.
#    Validation, test and blocking candidates are NOT used here.
#    Self-contained: defines every helper used later (sims, linear_score, ...).
#    Needs: FEATURE_COLUMNS, features (section 5) and PUB_NORM / add_pub_norm ("Publisher normalization").
# ============================================================
import itertools
import numpy as np
from scipy.optimize import nnls


def col(prefix):
    return next(c for c in FEATURE_COLUMNS if c.startswith(prefix))


# attributes of the rule (key "publisher_clean" kept for compatibility; similarity used = publisher_norm)
ATTRS = ["title", "first_author", "publisher_clean", "year", "page_count", "format_clean"]
SIM_COL = {"first_author": col("StringComparator(first_author, jaro_winkler"),
           "publisher_clean": PUB_NORM,
           "year": col("NumericComparator(year"),
           "page_count": col("NumericComparator(page_count"),
           "format_clean": col("StringComparator(format_clean")}
PRESENT_COL = {"publisher_clean": "BothPresent(publisher)", "year": "BothPresent(year)",
               "page_count": "BothPresent(page_count)", "format_clean": "BothPresent(format)"}
THRESHOLDS = np.round(np.arange(0.50, 1.00, 0.01), 2)
METHODS = ["grid 0-3", "grid 0-4", "grid 0-5", "least squares"]   # order = tie-break (simpler first)


def sims(df):
    """Similarities (S) and 'both values present' flags (P), one column per attribute."""
    title = np.maximum(df[col("StringComparator(title_clean, jaro_winkler")],
                       df[col("StringComparator(title_after_colon, jaro_winkler")])
    S = [title.to_numpy(float)] + [df[SIM_COL[a]].to_numpy(float) for a in ATTRS[1:]]
    P = [df[PRESENT_COL[a]].to_numpy(float) if a in PRESENT_COL else np.ones(len(df)) for a in ATTRS]
    return np.column_stack(S), np.column_stack(P)


def linear_score(S, P, w):
    """Weighted average over the attributes both records have."""
    w = np.asarray(w, float)
    num, den = (S * P * w).sum(1), (P * w).sum(1)
    return np.divide(num, den, out=np.zeros_like(num), where=den > 0)


def best_threshold(score, y):
    """Threshold with the highest F1 (used here only to score weight combinations on TRAIN)."""
    pred = score[None, :] >= THRESHOLDS[:, None]
    tp, fp, fn = (pred & y).sum(1), (pred & ~y).sum(1), (~pred & y).sum(1)
    f1 = np.where(tp > 0, 2 * tp / (2 * tp + fp + fn), 0)
    i = f1.argmax()
    return THRESHOLDS[i], f1[i]


def grid_search_train(S, P, y, max_w):
    """Title and author 1..max_w, other attributes 0..max_w; keep the weights with the best TRAIN F1."""
    ta, other = range(1, max_w + 1), range(0, max_w + 1)
    best = None
    for w in itertools.product(ta, ta, *[other] * (len(ATTRS) - 2)):
        _, f = best_threshold(linear_score(S, P, w), y)
        if best is None or f > best[1]:
            best = (w, f)
    return list(best[0]), best[1]


# ---------- learn the weights on the gold-standard TRAINING set ----------
weights = {m: {} for m in METHODS}
train_f1 = {m: {} for m in METHODS}
for p in features:
    tr = features[p]["train"]
    S_tr, P_tr = sims(tr)
    y_tr = tr["label"].to_numpy(bool)

    for max_w in [3, 4, 5]:                                    # grid search
        w, f = grid_search_train(S_tr, P_tr, y_tr, max_w)
        weights[f"grid 0-{max_w}"][p], train_f1[f"grid 0-{max_w}"][p] = w, f

    w_ls, _ = nnls(S_tr * P_tr, y_tr.astype(float))           # least squares (lecture, slide 74)
    weights["least squares"][p] = list(w_ls)
    _, train_f1["least squares"][p] = best_threshold(linear_score(S_tr, P_tr, w_ls), y_tr)
    print(p, "done")

NAMES = [a.replace("publisher_clean", "publisher_norm") for a in ATTRS]   # display names only

# table 1: grid search weights (integers, as learned) + train F1
grid_rows = []
for m in ["grid 0-3", "grid 0-4", "grid 0-5"]:
    for p, w in weights[m].items():
        grid_rows.append({"method": m, "pair": p, **dict(zip(NAMES, [int(x) for x in w])),
                          "train F1": round(train_f1[m][p], 3)})
print("=== Grid search: weights and F1, both on the gold-standard training set ===")
display(pd.DataFrame(grid_rows).set_index(["method", "pair"]))

# table 2: least-squares weights (continuous by nature) + train F1
ls_rows = [{"pair": p, **dict(zip(NAMES, np.round(w, 3))), "train F1": round(train_f1["least squares"][p], 3)}
           for p, w in weights["least squares"].items()]
print("=== Least squares: weights and F1, both on the gold-standard training set ===")
display(pd.DataFrame(ls_rows).set_index("pair"))

goodreads_amazon done
goodreads_dbpedia done
amazon_dbpedia done
=== Grid search: weights and F1, both on the gold-standard training set ===


title  first_author  publisher_norm  year  \
method   pair                                                           
grid 0-3 goodreads_amazon       1             1               1     1   
         goodreads_dbpedia      2             1               2     3   
         amazon_dbpedia         3             1               2     2   
grid 0-4 goodreads_amazon       1             1               1     1   
         goodreads_dbpedia      2             1               2     4   
         amazon_dbpedia         4             1               2     2   
grid 0-5 goodreads_amazon       1             1               1     1   
         goodreads_dbpedia      1             1               2     5   
         amazon_dbpedia         5             1               3     3   

                            page_count  format_clean  train F1  
method   pair                                                   
grid 0-3 goodreads_amazon            3             2     0.899  
         goodreads_dbpedia           3             0     0.771  
         amazon_dbpedia              1             0     0.770  
grid 0-4 goodreads_amazon            4             2     0.908  
         goodreads_dbpedia           4             0     0.773  
         amazon_dbpedia              1             0     0.772  
grid 0-5 goodreads_amazon            4             2     0.908  
         goodreads_dbpedia           5             0     0.776  
         amazon_dbpedia              3             0     0.774

=== Least squares: weights and F1, both on the gold-standard training set ===


,title,first_author,publisher_norm,year,page_count,format_clean,train F1
pair,,,,,,,
goodreads_amazon,0.000,0.0,0.504,0.138,0.238,0.003,0.807
goodreads_dbpedia,0.014,0.0,0.394,0.241,0.187,0.000,0.745
amazon_dbpedia,0.000,0.0,0.376,0.199,0.237,0.000,0.732


In [ ]:
# ============================================================
# 2) VALIDATION: end-to-end with 1:1
#    Blocking candidates -> score (frozen weights from cell 1) -> threshold -> greedy 1:1
#    F1 is computed against the gold-standard VALIDATION set.
#    Computes validation F1 for every method and threshold. The test set is NOT used here.
# ============================================================
GRID = [0.5, 0.6, 0.7, 0.8, 0.9]


def chunks(df, size=100_000):
    for i in range(0, len(df), size):
        yield df.iloc[i:i + size]


# comparators for every similarity except publisher_norm (added below with add_pub_norm)
needed = ({col("StringComparator(title_clean, jaro_winkler"), col("StringComparator(title_after_colon, jaro_winkler")}
          | set(SIM_COL.values()) | set(PRESENT_COL.values())) - {PUB_NORM}
extractor = FeatureExtractor([c for c in comparators if c.name in needed])

rows = []
for a, b in PAIRS:
    p = f"{a}_{b}"
    feats = pd.concat([extractor.create_features(data[a], data[b], c, id_column="id")
                       for c in chunks(candidates[p])], ignore_index=True)
    feats = add_pub_norm(feats, a, b)              # same publisher_norm similarity as for the gold pairs
    S_c, P_c = sims(feats)
    for method in METHODS:
        sc = feats[["id1", "id2"]].copy()
        sc["score"] = linear_score(S_c, P_c, weights[method][p])
        for thr in GRID:
            c = GreedyOneToOneMatchingAlgorithm().cluster(sc[sc["score"] >= thr].reset_index(drop=True))
            ev = EntityMatchingEvaluator.evaluate_matching(correspondences=c, test_pairs=gold[p]["validation"])
            rows.append({"pair": p, "method": method, "threshold": thr, "correspondences": len(c),
                         "precision": ev["precision"], "recall": ev["recall"], "f1": ev["f1"]})
    print(p, "done")

val_cmp = pd.DataFrame(rows)
val_cmp.to_csv(OUT_DIR / "method_comparison_validation_1to1.csv", index=False)

# all F1 values: one table per pair (methods x thresholds)
for p in val_cmp["pair"].unique():
    print(f"\n=== {p}: validation F1 with 1:1 (gold-standard validation set) ===")
    display(val_cmp[val_cmp.pair == p].pivot(index="method", columns="threshold", values="f1")
            .reindex(METHODS).round(3))

goodreads_amazon done
goodreads_dbpedia done
amazon_dbpedia done

=== goodreads_amazon: validation F1 with 1:1 (gold-standard validation set) ===


threshold,0.5,0.6,0.7,0.8,0.9
method,,,,,
grid 0-3,0.793,0.789,0.804,0.807,0.673
grid 0-4,0.793,0.793,0.783,0.789,0.724
grid 0-5,0.793,0.793,0.783,0.789,0.724
least squares,0.522,0.547,0.538,0.538,0.516



=== goodreads_dbpedia: validation F1 with 1:1 (gold-standard validation set) ===


threshold,0.5,0.6,0.7,0.8,0.9
method,,,,,
grid 0-3,0.697,0.713,0.738,0.677,0.440
grid 0-4,0.701,0.713,0.733,0.725,0.440
grid 0-5,0.711,0.720,0.740,0.721,0.473
least squares,0.692,0.618,0.588,0.459,0.430



=== amazon_dbpedia: validation F1 with 1:1 (gold-standard validation set) ===


threshold,0.5,0.6,0.7,0.8,0.9
method,,,,,
grid 0-3,0.568,0.571,0.553,0.538,0.395
grid 0-4,0.614,0.612,0.586,0.539,0.410
grid 0-5,0.547,0.559,0.565,0.538,0.359
least squares,0.532,0.500,0.472,0.442,0.359


In [ ]:
# ============================================================
# 3) Choose method + threshold PER PAIR: highest validation F1 (with 1:1)
#    Each pair has its own train/validation/test split (separate matching task).
#    Uses the results of cell 2 (no recomputation). The test set is NOT used here.
# ============================================================
try:
    val_cmp
except NameError:                                   # e.g. after a restart
    val_cmp = pd.read_csv(OUT_DIR / "method_comparison_validation_1to1.csv")

METHODS = ["grid 0-3", "grid 0-4", "grid 0-5", "least squares"]   # order = tie-break (simpler first)

# best F1 per method and pair (ties -> stricter threshold)
best = (val_cmp.sort_values("threshold", ascending=False)
               .loc[lambda d: d.groupby(["pair", "method"])["f1"].idxmax()])
summary = best.pivot(index="method", columns="pair", values="f1").reindex(METHODS)
summary["mean"] = summary.mean(axis=1)
print("\n=== Best validation F1 per method (with 1:1) ===")
display(summary.round(3))

# threshold of that best F1
print("=== Threshold of the best F1 ===")
display(best.pivot(index="method", columns="pair", values="threshold").reindex(METHODS))

# chosen per pair: highest validation F1 (ties -> simpler method)
best["order"] = best["method"].map(METHODS.index)
final_choice = (best.sort_values(["pair", "f1", "order"], ascending=[True, False, True])
                    .groupby("pair").head(1)
                    [["pair", "method", "threshold", "precision", "recall", "f1", "correspondences"]]
                    .reset_index(drop=True))

# add the training F1 of the chosen method (from the train cell; gold-standard training set)
final_choice.insert(3, "train f1", [train_f1[m][p] for p, m in zip(final_choice["pair"], final_choice["method"])])
final_choice = final_choice.rename(columns={"precision": "val precision", "recall": "val recall", "f1": "val f1",
                                            "correspondences": "val correspondences"})
print("=== Chosen on validation (per pair) ===")
display(final_choice.round(3))

# ------------------------------------------------------------
# Train vs validation, without and with 1:1 (for reporting only, no decision is made here)
#   without 1:1: pairwise on the gold-standard validation pairs, same weights,
#                its own best threshold on the 0.50-0.99 grid
#   with 1:1:    end-to-end result chosen above
# ------------------------------------------------------------
val_no11, thr_no11 = [], []
for p, m in zip(final_choice["pair"], final_choice["method"]):
    va = features[p]["validation"]
    S_va, P_va = sims(va)
    t, f = best_threshold(linear_score(S_va, P_va, weights[m][p]), va["label"].to_numpy(bool))
    thr_no11.append(t)
    val_no11.append(f)

comparison = pd.DataFrame({
    "pair": final_choice["pair"],
    "method": final_choice["method"],
    "train f1": final_choice["train f1"],
    "val f1 (without 1:1)": val_no11,
    "val f1 (with 1:1)": final_choice["val f1"],
})
print("=== Train vs validation (without and with 1:1) ===")
display(comparison.round(3))


=== Best validation F1 per method (with 1:1) ===


pair,amazon_dbpedia,goodreads_amazon,goodreads_dbpedia,mean
method,,,,
grid 0-3,0.571,0.807,0.738,0.705
grid 0-4,0.614,0.793,0.733,0.713
grid 0-5,0.565,0.793,0.740,0.700
least squares,0.532,0.547,0.692,0.591


=== Threshold of the best F1 ===


pair,amazon_dbpedia,goodreads_amazon,goodreads_dbpedia
method,,,
grid 0-3,0.6,0.8,0.7
grid 0-4,0.5,0.6,0.7
grid 0-5,0.7,0.6,0.7
least squares,0.5,0.6,0.5


=== Chosen on validation (per pair) ===


,pair,method,threshold,train f1,val precision,val recall,val f1,val correspondences
0,amazon_dbpedia,grid 0-4,0.5,0.772,0.705,0.544,0.614,9645
1,goodreads_amazon,grid 0-3,0.8,0.899,0.979,0.687,0.807,5107
2,goodreads_dbpedia,grid 0-5,0.7,0.776,0.679,0.814,0.740,11572


=== Train vs validation (without and with 1:1) ===


,pair,method,train f1,val f1 (without 1:1),val f1 (with 1:1)
0,amazon_dbpedia,grid 0-4,0.772,0.775,0.614
1,goodreads_amazon,grid 0-3,0.899,0.886,0.807
2,goodreads_dbpedia,grid 0-5,0.776,0.736,0.740


### 6.1b Non Linear Matching Rule

In [ ]:
# Error analysis of MY weighted linear rule on the VALIDATION set (without 1:1).
# Method per pair = the one chosen on validation; threshold = best validation F1.
# Counts which factors appear among FP / FN -> conditions of the non-linear rule.

CHOSEN = {"amazon_dbpedia": "grid 0-4", "goodreads_amazon": "grid 0-3", "goodreads_dbpedia": "grid 0-5"}

TITLE, TITLE_AC = col("StringComparator(title_clean, jaro_winkler"), col("StringComparator(title_after_colon, jaro_winkler")
AUTHOR, YEAR = col("StringComparator(first_author, jaro_winkler"), col("NumericComparator(year")
PUB, PAGES, FMT = PUB_NORM, col("NumericComparator(page_count"), col("StringComparator(format_clean")


def n_missing(r):
    """Number of edition attributes not present in both records."""
    return 4 - int(r["BothPresent(publisher)"] + r["BothPresent(year)"]
                   + r["BothPresent(page_count)"] + r["BothPresent(format)"])


# "differs" = both records have a value and it is different
FACTORS = {
    "title similar (>= 0.9)":         lambda r: max(r[TITLE], r[TITLE_AC]) >= 0.9,
    "author similar (>= 0.9)":        lambda r: r[AUTHOR] >= 0.9,
    "title differs (< 0.9)":          lambda r: max(r[TITLE], r[TITLE_AC]) < 0.9,
    "author differs (< 0.9)":         lambda r: r[AUTHOR] < 0.9,
    "year differs (< 0.9)":           lambda r: r["BothPresent(year)"] == 1 and r[YEAR] < 0.9,
    "publisher_norm differs (< 0.5)": lambda r: r["BothPresent(publisher)"] == 1 and r[PUB] < 0.5,
    "page_count differs (< 0.9)":     lambda r: r["BothPresent(page_count)"] == 1 and r[PAGES] < 0.9,
    "format differs (< 0.9)":         lambda r: r["BothPresent(format)"] == 1 and r[FMT] < 0.9,
    "2+ edition attributes missing":  lambda r: n_missing(r) >= 2,
}
RAW_COLS = ["title", "first_author", "publisher", "year", "page_count", "format"]

factor_table, val_errors = {}, {}
for p in features:
    va = features[p]["validation"].copy()
    y = va["label"].to_numpy(bool)

    # apply my linear rule to the validation pairs
    S, P = sims(va)
    va["score"] = linear_score(S, P, weights[CHOSEN[p]][p])
    thr, _ = best_threshold(va["score"].to_numpy(), y)        # = validation threshold without 1:1
    pred = va["score"].to_numpy() >= thr

    fp, fn = va[pred & ~y], va[~pred & y]
    val_errors[p] = {"FP": fp, "FN": fn}

    # share of errors with each factor (one error can have several)
    c = {("", "method"): CHOSEN[p], ("", "threshold (val, no 1:1)"): thr,
         ("FP", "total"): len(fp), ("FN", "total"): len(fn)}
    for kind, err in [("FP", fp), ("FN", fn)]:
        for name, f in FACTORS.items():
            n = sum(bool(f(r)) for _, r in err.iterrows())
            c[(kind, name)] = f"{n} ({n / len(err):.0%})" if len(err) else "0"
    factor_table[p] = c

print("=== Error factors of my weighted linear rule on VALIDATION (without 1:1) ===")
display(pd.DataFrame(factor_table))

# example errors with raw values
pd.set_option("display.max_colwidth", 40)
for p, errs in val_errors.items():
    a, b = p.split("_")
    A, B = data[a].set_index("id"), data[b].set_index("id")
    for kind in ["FP", "FN"]:
        rows = [{"score": round(s, 3),
                 **{f"{c} ({a})": A.at[x, c] for c in RAW_COLS},
                 **{f"{c} ({b})": B.at[y_, c] for c in RAW_COLS}}
                for x, y_, s in zip(errs[kind]["id1"], errs[kind]["id2"], errs[kind]["score"])]
        print(f"\n=== {p}: first 5 {kind} on VALIDATION ===")
        display(pd.DataFrame(rows).head(5))

=== Error factors of my weighted linear rule on VALIDATION (without 1:1) ===


goodreads_amazon goodreads_dbpedia  \
   method                                 grid 0-3          grid 0-5   
   threshold (val, no 1:1)                    0.82               0.7   
FP total                                        11                40   
FN total                                         5                 6   
FP title similar (>= 0.9)                 10 (91%)          35 (88%)   
   author similar (>= 0.9)                10 (91%)          38 (95%)   
   title differs (< 0.9)                    1 (9%)           5 (12%)   
   author differs (< 0.9)                   1 (9%)            2 (5%)   
   year differs (< 0.9)                    3 (27%)           5 (12%)   
   publisher_norm differs (< 0.5)           1 (9%)          18 (45%)   
   page_count differs (< 0.9)               1 (9%)          16 (40%)   
   format differs (< 0.9)                  7 (64%)            0 (0%)   
   2+ edition attributes missing            1 (9%)          19 (48%)   
FN title similar (>= 0.9)                  3 (60%)           2 (33%)   
   author similar (>= 0.9)                 4 (80%)           4 (67%)   
   title differs (< 0.9)                   2 (40%)           4 (67%)   
   author differs (< 0.9)                  1 (20%)           2 (33%)   
   year differs (< 0.9)                    1 (20%)           5 (83%)   
   publisher_norm differs (< 0.5)          4 (80%)           2 (33%)   
   page_count differs (< 0.9)              4 (80%)           3 (50%)   
   format differs (< 0.9)                   0 (0%)            0 (0%)   
   2+ edition attributes missing            0 (0%)            0 (0%)   

                                  amazon_dbpedia  
   method                               grid 0-4  
   threshold (val, no 1:1)                  0.78  
FP total                                      22  
FN total                                       7  
FP title similar (>= 0.9)              22 (100%)  
   author similar (>= 0.9)             22 (100%)  
   title differs (< 0.9)                  0 (0%)  
   author differs (< 0.9)                 0 (0%)  
   year differs (< 0.9)                   2 (9%)  
   publisher_norm differs (< 0.5)         2 (9%)  
   page_count differs (< 0.9)            4 (18%)  
   format differs (< 0.9)                 0 (0%)  
   2+ edition attributes missing        18 (82%)  
FN title similar (>= 0.9)                3 (43%)  
   author similar (>= 0.9)               4 (57%)  
   title differs (< 0.9)                 4 (57%)  
   author differs (< 0.9)                3 (43%)  
   year differs (< 0.9)                  3 (43%)  
   publisher_norm differs (< 0.5)        4 (57%)  
   page_count differs (< 0.9)            1 (14%)  
   format differs (< 0.9)                 0 (0%)  
   2+ edition attributes missing         4 (57%)


=== goodreads_amazon: first 5 FP on VALIDATION ===


,score,title (goodreads),first_author (goodreads),publisher (goodreads),year (goodreads),page_count (goodreads),format (goodreads),title (amazon),first_author (amazon),publisher (amazon),year (amazon),page_count (amazon),format (amazon)
0,0.855,The Innocent,David Baldacci,Grand Central Publishing,2012,422.0,Hardcover,"The Innocent (Will Robie Series, 1)",David Baldacci,Grand Central Publishing,2012,NaN,Audio CD
1,0.886,Ten Big Ones,Janet Evanovich,St. Martin's Press,2005,319.0,Mass Market Paperback,Ten Big Ones (Stephanie Plum Novels),Janet Evanovich,St. Martin's Press,2004,320.0,Hardcover
2,0.890,The Nest,Cynthia D'Aprix Sweeney,Ecco,2016,368.0,Hardcover,The Nest,Cynthia D'Aprix Sweeney,Ecco,2017,368.0,Paperback
3,0.825,The Wednesday Letters,Jason F. Wright,Shadow Mountain,2007,280.0,Hardcover,The Rent Collector,Camron Wright,Shadow Mountain,2012,304.0,Hardcover
4,1.000,The Black Widow,Daniel Silva,Harper,2016,528.0,Hardcover,The Black Widow (Signed Book) (Gabri...,Daniel Silva,NaN,<NA>,NaN,NaN



=== goodreads_amazon: first 5 FN on VALIDATION ===


,score,title (goodreads),first_author (goodreads),publisher (goodreads),year (goodreads),page_count (goodreads),format (goodreads),title (amazon),first_author (amazon),publisher (amazon),year (amazon),page_count (amazon),format (amazon)
0,0.778,Where's Spot?,Eric Hill,Puffin,1983,24.0,Paperback,Where's Spot?,Eric Hill,Gardners Books,1983,20.0,Paperback
1,0.749,Die Herren von Winterfell,George R.R. Martin,Blanvalet,2010,545.0,Paperback,Das Lied von Eis und Feuer 01. Die H...,George R. R. Martin,French and European Publications Inc,2010,576.0,NaN
2,0.762,War Brides,Helen Bryan,Penguin Group,2007,467.0,Paperback,War Brides,Helen Bryan,Penguin Books,2007,300.0,Paperback
3,0.684,Heart of Darkness: and Selections fr...,Joseph Conrad,Modern Library,1999,162.0,Paperback,Heart of Darkness and The Congo Diar...,Joseph Conrad,Penguin Classics,2007,192.0,Paperback
4,0.734,The World of Ice and Fire: The Untol...,George R.R. Martin,HarperVoyager UK,2014,326.0,Hardcover,The World of Ice and Fire: The Untol...,Linda Antonsson,HARPER COLLINS,2014,NaN,Hardcover


... (4 outputs left out)


=== amazon_dbpedia: first 5 FP on VALIDATION ===


,score,title (amazon),first_author (amazon),publisher (amazon),year (amazon),page_count (amazon),format (amazon),title (dbpedia),first_author (dbpedia),publisher (dbpedia),year (dbpedia),page_count (dbpedia),format (dbpedia)
0,0.889,The Hanging Tree: The Sixth Rivers o...,Ben Aaronovitch,Gollancz,2016,NaN,Audio CD,The Hanging Tree (Aaronovitch novel),Ben Aaronovitch,Victor Gollancz Ltd,2016,NaN,None
1,1.000,Kiss the Girls,James Patterson,Little Brown,1995,NaN,NaN,Kiss the Girls (novel),James Patterson,"Little, Brown and Company",1995,464.0,None
2,1.000,The Luminaries: A Novel (Man Booker ...,Eleanor Catton,"Little, Brown and Company",2013,848.0,Hardcover,The Luminaries,Eleanor Catton,NaN,<NA>,848.0,None
3,1.000,Shade: A Tale of Two Presidents,Pete Souza,"Little, Brown and Company",2018,240.0,Hardcover,Shade: A Tale of Two Presidents,Pete Souza,"Little, Brown and Company",2018,NaN,None
4,1.000,One for the Money,Janet Evanovich,NaN,<NA>,NaN,Paperback,One for the Money (novel),Janet Evanovich,St. Martin's Press,1994,288.0,None



=== amazon_dbpedia: first 5 FN on VALIDATION ===


,score,title (amazon),first_author (amazon),publisher (amazon),year (amazon),page_count (amazon),format (amazon),title (dbpedia),first_author (dbpedia),publisher (dbpedia),year (dbpedia),page_count (dbpedia),format (dbpedia)
0,0.778,On Wings of Eagles,Ken Follett,William Morrow & Co,1983,444.0,Hardcover,On Wings of Eagles,Ken Follett,HarperCollins,1983,NaN,None
1,0.461,Alcoholics Anonymous: The Story of H...,Alcoholics Anonymous World Service Inc.,Alcoholics Anonymous World Services,2003,575.0,Hardcover,The Big Book (Alcoholics Anonymous),Bill W.,Alcoholics Anonymous World Services,1955,NaN,None
2,0.490,Crucial Conversations Tools for Talk...,Joseph Grenny,McGraw Hill,2011,272.0,Paperback,Crucial Conversations: Tools for Tal...,Al Switzler,GradeGuru,2002,288.0,None
3,0.756,Consent to Kill: A Thriller (8) (A M...,Vince Flynn,Pocket Star,2006,704.0,Mass Market Paperback,Consent to Kill,Vince Flynn,Simon & Schuster,2005,NaN,None
4,0.639,La Sombra del Viento (Spanish Edition),Carlos Ruiz Zafón,Planeta,2006,567.0,Paperback,The Shadow of the Wind,Carlos Ruiz Zafón,Grupo Planeta,2001,565.0,None


In [ ]:
# ============================================================
# NON-LINEAR RULE (lecture, slide 13) – A) TRAIN
# Cascade: 1) gate on title + author  2) exclusion if format / pages / publisher /
#          year clearly differ (only if present in both)  3) at most one edition
#          attribute present -> title + author must be very similar  4) otherwise match.
# Thresholds are learned on the gold-standard TRAIN pairs, per pair and per variant.
# ============================================================
import itertools
import numpy as np
import pandas as pd


def nonlinear_pred(S, P, th):
    """Match decision of the cascade. S, P from sims(); th = dict of thresholds."""
    t, a, pub, yr, pg, fmt = S.T                     
    _, _, Ppub, Pyr, Ppg, Pfmt = P.T                
    gate = (t >= th["title"]) & (a >= th["author"])                          
    excluded = (((Pfmt == 1) & (fmt < th["format"]))                          
                | ((Ppg == 1) & (pg < th["pages"]))
                | ((Ppub == 1) & (pub < th["publisher"]))
                | ((Pyr == 1) & (yr < th["year"])))                           
    no_edition = (Ppub + Ppg + Pfmt) <= 1                                    
    strict_ok = (t >= th["strict"]) & (a >= th["strict"])
    return gate & ~excluded & (~no_edition | strict_ok)                       


def f1(y, pred):
    tp, fp, fn = (pred & y).sum(), (pred & ~y).sum(), (~pred & y).sum()
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0


# threshold 0 = condition switched off
GRID = {"title":     [0.70, 0.80, 0.85, 0.90, 0.95],
        "author":    [0.70, 0.80, 0.90, 0.95],
        "format":    [0.0, 0.5, 0.7, 0.9],
        "pages":     [0.0, 0.3, 0.5, 0.7, 0.9],
        "publisher": [0.0, 0.2, 0.4, 0.6],
        "year":      [0.0, 0.5, 0.7, 0.9],          # CHANGED: new
        "strict":    [0.0, 0.90, 0.95, 0.98]}

# ablation: switched-off conditions are fixed to 0 (order = tie-break, simpler first)
VARIANTS = {"gate only":            {"format": [0.0], "pages": [0.0], "publisher": [0.0], "year": [0.0], "strict": [0.0]},
            "without exclusion":    {"format": [0.0], "pages": [0.0], "publisher": [0.0], "year": [0.0]},
            "without missing-data": {"strict": [0.0]},
            "without format":       {"format": [0.0]},
            "full":                 {}}

nl_th, nl_train_f1 = {}, {}
for p in features:
    tr = features[p]["train"]
    S_tr, P_tr = sims(tr)
    y_tr = tr["label"].to_numpy(bool)
    for v, fixed in VARIANTS.items():
        grid = {k: fixed.get(k, vals) for k, vals in GRID.items()}
        best = None
        for combo in itertools.product(*grid.values()):
            th = dict(zip(grid, combo))
            f = f1(y_tr, nonlinear_pred(S_tr, P_tr, th))
            if best is None or f > best[1]:
                best = (th, f)
        nl_th[p, v], nl_train_f1[p, v] = best
    print(p, "done")

rows = [{"pair": p, "variant": v, **th, "train F1": round(nl_train_f1[p, v], 3)} for (p, v), th in nl_th.items()]
print("=== Non-linear rule: thresholds and F1, both on the gold-standard TRAIN set ===")
display(pd.DataFrame(rows).set_index(["pair", "variant"]))

goodreads_amazon done
goodreads_dbpedia done
amazon_dbpedia done
=== Non-linear rule: thresholds and F1, both on the gold-standard TRAIN set ===


title  author  format  pages  \
pair              variant                                              
goodreads_amazon  gate only               0.8     0.9     0.0    0.0   
                  without exclusion       0.8     0.9     0.0    0.0   
                  without missing-data    0.7     0.7     0.7    0.0   
                  without format          0.7     0.7     0.0    0.3   
                  full                    0.7     0.7     0.7    0.0   
goodreads_dbpedia gate only               0.7     0.8     0.0    0.0   
                  without exclusion       0.7     0.8     0.0    0.0   
                  without missing-data    0.7     0.8     0.0    0.7   
                  without format          0.7     0.8     0.0    0.7   
                  full                    0.7     0.8     0.0    0.7   
amazon_dbpedia    gate only               0.8     0.8     0.0    0.0   
                  without exclusion       0.8     0.8     0.0    0.0   
                  without missing-data    0.7     0.8     0.0    0.0   
                  without format          0.7     0.8     0.0    0.0   
                  full                    0.7     0.8     0.0    0.0   

                                        publisher  year  strict  train F1  
pair              variant                                                  
goodreads_amazon  gate only                   0.0   0.0    0.00     0.636  
                  without exclusion           0.0   0.0    0.98     0.638  
                  without missing-data        0.2   0.0    0.00     0.828  
                  without format              0.2   0.9    0.95     0.814  
                  full                        0.2   0.0    0.95     0.831  
goodreads_dbpedia gate only                   0.0   0.0    0.00     0.594  
                  without exclusion           0.0   0.0    0.00     0.594  
                  without missing-data        0.0   0.9    0.00     0.685  
                  without format              0.0   0.9    0.00     0.685  
                  full                        0.0   0.9    0.00     0.685  
amazon_dbpedia    gate only                   0.0   0.0    0.00     0.567  
                  without exclusion           0.0   0.0    0.00     0.567  
                  without missing-data        0.2   0.9    0.00     0.687  
                  without format              0.2   0.9    0.00     0.687  
                  full                        0.2   0.9    0.00     0.687

In [ ]:
# ============================================================
# NON-LINEAR RULE – B) VALIDATION without 1:1 (for reporting, no decision here)
# Compares every variant with the linear rule on the validation pairs and
# shows train vs validation F1 (overfitting check).
# ============================================================
from sklearn.metrics import precision_score, recall_score

CHOSEN = dict(zip(final_choice["pair"], final_choice["method"]))

val_rows = []
for p in features:
    va = features[p]["validation"]
    S_va, P_va = sims(va)
    y_va = va["label"].to_numpy(bool)

    lin_score = linear_score(S_va, P_va, weights[CHOSEN[p]][p])
    lin_thr, _ = best_threshold(lin_score, y_va)
    lin_pred = lin_score >= lin_thr
    val_rows.append({"pair": p, "variant": "LINEAR RULE (reference)", "train F1": round(train_f1[CHOSEN[p]][p], 3),
                     "val precision": round(precision_score(y_va, lin_pred, zero_division=0), 3),
                     "val recall": round(recall_score(y_va, lin_pred), 3), "val F1": round(f1(y_va, lin_pred), 3)})

    for v in VARIANTS:
        pred = nonlinear_pred(S_va, P_va, nl_th[p, v])
        val_rows.append({"pair": p, "variant": v, "train F1": round(nl_train_f1[p, v], 3),
                         "val precision": round(precision_score(y_va, pred, zero_division=0), 3),
                         "val recall": round(recall_score(y_va, pred), 3), "val F1": round(f1(y_va, pred), 3)})

print("=== Validation without 1:1: non-linear variants vs linear rule ===")
display(pd.DataFrame(val_rows).set_index(["pair", "variant"]))

=== Validation without 1:1: non-linear variants vs linear rule ===


train F1  val precision  \
pair              variant                                            
goodreads_amazon  LINEAR RULE (reference)     0.899          0.849   
                  gate only                   0.636          0.457   
                  without exclusion           0.638          0.467   
                  without missing-data        0.828          0.920   
                  without format              0.814          0.746   
                  full                        0.831          0.920   
goodreads_dbpedia LINEAR RULE (reference)     0.776          0.615   
                  gate only                   0.594          0.453   
                  without exclusion           0.594          0.453   
                  without missing-data        0.685          0.608   
                  without format              0.685          0.608   
                  full                        0.685          0.608   
amazon_dbpedia    LINEAR RULE (reference)     0.772          0.694   
                  gate only                   0.567          0.439   
                  without exclusion           0.567          0.439   
                  without missing-data        0.687          0.672   
                  without format              0.687          0.672   
                  full                        0.687          0.672   

                                           val recall  val F1  
pair              variant                                      
goodreads_amazon  LINEAR RULE (reference)       0.925   0.886  
                  gate only                     0.940   0.615  
                  without exclusion             0.940   0.624  
                  without missing-data          0.687   0.786  
                  without format                0.701   0.723  
                  full                          0.687   0.786  
goodreads_dbpedia LINEAR RULE (reference)       0.914   0.736  
                  gate only                     0.757   0.567  
                  without exclusion             0.757   0.567  
                  without missing-data          0.686   0.644  
                  without format                0.686   0.644  
                  full                          0.686   0.644  
amazon_dbpedia    LINEAR RULE (reference)       0.877   0.775  
                  gate only                     0.877   0.585  
                  without exclusion             0.877   0.585  
                  without missing-data          0.684   0.678  
                  without format                0.684   0.678  
                  full                          0.684   0.678

In [ ]:
# ============================================================
# NON-LINEAR RULE – C) VALIDATION with 1:1: choose variant + 1:1 threshold per pair
# The rule filters all blocking candidates; pairs that pass keep the linear-rule
# score, so greedy 1:1 can rank competing candidates. Best validation F1 wins
# (ties -> simpler variant, then stricter threshold). Test set is NOT used.
# ============================================================
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm
from PyDI.entitymatching.evaluation import EntityMatchingEvaluator


def chunks(df, size=100_000):
    for i in range(0, len(df), size):
        yield df.iloc[i:i + size]


# features of all blocking candidates (computed once, kept in cand_feats)
NEEDED = [col("StringComparator(title_clean, jaro_winkler"), col("StringComparator(title_after_colon, jaro_winkler")] \
         + [c for c in SIM_COL.values() if c != PUB_NORM] + list(PRESENT_COL.values())
rule_extractor = FeatureExtractor([c for c in comparators if c.name in NEEDED])
try:
    cand_feats
except NameError:
    cand_feats = {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    if p not in cand_feats:
        feats = pd.concat([rule_extractor.create_features(data[a], data[b], ch, id_column="id")
                           for ch in chunks(candidates[p])], ignore_index=True)
        cand_feats[p] = add_pub_norm(feats, a, b)
        print(p, f"{len(feats):,} candidates")

TH_11 = np.round(np.r_[0.0, np.arange(0.50, 1.00, 0.05)], 2)     # 0 = keep every pair accepted by the rule
order = {v: i for i, v in enumerate(VARIANTS)}
all_rows = []
for p, feats in cand_feats.items():
    S_c, P_c = sims(feats)
    lin = linear_score(S_c, P_c, weights[CHOSEN[p]][p])
    for v in VARIANTS:
        keep = nonlinear_pred(S_c, P_c, nl_th[p, v])
        sc = pd.DataFrame({"id1": feats["id1"], "id2": feats["id2"], "score": lin})[keep]
        for thr in TH_11:
            c = GreedyOneToOneMatchingAlgorithm().cluster(sc[sc["score"] >= thr].reset_index(drop=True))
            ev = EntityMatchingEvaluator.evaluate_matching(correspondences=c, test_pairs=gold[p]["validation"])
            all_rows.append({"pair": p, "variant": v, "threshold": thr, "precision": ev["precision"],
                             "recall": ev["recall"], "f1": ev["f1"], "correspondences": len(c)})

nl_val_11 = pd.DataFrame(all_rows)
nl_val_11["order"] = nl_val_11["variant"].map(order)
nl_final = (nl_val_11.sort_values(["pair", "f1", "order", "threshold"], ascending=[True, False, True, False])
                     .groupby("pair").head(1).drop(columns="order").reset_index(drop=True))

print("=== Best validation F1 (with 1:1) per variant ===")
display(nl_val_11.loc[nl_val_11.groupby(["pair", "variant"])["f1"].idxmax()]
                 .pivot(index="variant", columns="pair", values="f1").reindex(list(VARIANTS)).round(3))

LIN_VAL_11 = {"amazon_dbpedia": 0.614, "goodreads_amazon": 0.807, "goodreads_dbpedia": 0.740}   # linear rule, with 1:1
nl_final["linear rule val F1 (1:1)"] = nl_final["pair"].map(LIN_VAL_11)
print("=== Chosen on validation (per pair) ===")
display(nl_final.round(3))

goodreads_amazon 1,320,641 candidates
goodreads_dbpedia 1,074,254 candidates
amazon_dbpedia 1,216,635 candidates
=== Best validation F1 (with 1:1) per variant ===


pair,amazon_dbpedia,goodreads_amazon,goodreads_dbpedia
variant,,,
gate only,0.568,0.800,0.658
without exclusion,0.568,0.800,0.658
without missing-data,0.552,0.712,0.662
without format,0.552,0.724,0.662
full,0.552,0.712,0.662


=== Chosen on validation (per pair) ===


,pair,variant,threshold,precision,recall,f1,correspondences,linear rule val F1 (1:1)
0,amazon_dbpedia,gate only,0.0,0.711,0.474,0.568,2703,0.614
1,goodreads_amazon,gate only,0.7,0.958,0.687,0.800,3414,0.807
2,goodreads_dbpedia,without missing-data,0.7,0.640,0.686,0.662,5545,0.740


In [ ]:
# Sanity check: no conditions -> must reproduce the linear rule with 1:1 (~0.614 / 0.807 / 0.740)
for p, feats in cand_feats.items():
    S_c, P_c = sims(feats)
    sc = pd.DataFrame({"id1": feats["id1"], "id2": feats["id2"],
                       "score": linear_score(S_c, P_c, weights[CHOSEN[p]][p])})
    best = max((EntityMatchingEvaluator.evaluate_matching(
                    correspondences=GreedyOneToOneMatchingAlgorithm().cluster(sc[sc["score"] >= t].reset_index(drop=True)),
                    test_pairs=gold[p]["validation"])["f1"], t) for t in TH_11[1:])
    print(f"{p:<20} best F1 {best[0]:.3f} at threshold {best[1]}   (linear rule: {LIN_VAL_11[p]})")

goodreads_amazon     best F1 0.807 at threshold 0.8   (linear rule: 0.807)
goodreads_dbpedia    best F1 0.740 at threshold 0.7   (linear rule: 0.74)
amazon_dbpedia       best F1 0.614 at threshold 0.5   (linear rule: 0.614)


In [ ]:
# Sanity check for the non-linear rule (validation):
# (1) same decision on the validation pairs, whether features come from the gold-standard
#     table (used for training) or from the candidate table (cell C) -> should be 100%
# (2) how many TRUE validation matches the rule removes before 1:1, and at which step
for _, r in nl_final.iterrows():
    p, th = r["pair"], nl_th[r["pair"], r["variant"]]
    va, cf = features[p]["validation"], cand_feats[p]

    # (1) consistency
    keep_va = pd.Series(nonlinear_pred(*sims(va), th), index=pd.MultiIndex.from_frame(va[["id1", "id2"]]))
    keep_cf = pd.Series(nonlinear_pred(*sims(cf), th), index=pd.MultiIndex.from_frame(cf[["id1", "id2"]]))
    keep_cf = keep_cf[~keep_cf.index.duplicated()]
    common = keep_va.index.intersection(keep_cf.index)
    agree = (keep_va[common].to_numpy() == keep_cf[common].to_numpy()).mean()

    # (2) where are the true matches removed?
    S, P = sims(va)
    y = va["label"].to_numpy(bool)
    t, a, pub, yr, pg, fmt = S.T
    _, _, Ppub, Pyr, Ppg, Pfmt = P.T
    gate_fail = ~((t >= th["title"]) & (a >= th["author"]))
    excl = (((Pfmt == 1) & (fmt < th["format"])) | ((Ppg == 1) & (pg < th["pages"]))
            | ((Ppub == 1) & (pub < th["publisher"])) | ((Pyr == 1) & (yr < th["year"])))
    strict_fail = ((Ppub + Ppg + Pfmt) <= 1) & ~((t >= th["strict"]) & (a >= th["strict"]))

    print(f"\n=== {p} ({r['variant']}) ===")
    print(f"(1) same decision on {len(common)} validation pairs: {agree:.1%}")
    print(f"(2) true validation matches: {y.sum()}")
    print(f"    removed by gate (title/author):   {(gate_fail & y).sum()}")
    print(f"    removed by exclusion conditions:  {(~gate_fail & excl & y).sum()}")
    print(f"    removed by step 3 (missing data): {(~gate_fail & ~excl & strict_fail & y).sum()}")
    print(f"    kept for 1:1:                     {(~gate_fail & ~excl & ~strict_fail & y).sum()}")


=== amazon_dbpedia (gate only) ===
(1) same decision on 197 validation pairs: 100.0%
(2) true validation matches: 57
    removed by gate (title/author):   7
    removed by exclusion conditions:  0
    removed by step 3 (missing data): 0
    kept for 1:1:                     50

=== goodreads_amazon (gate only) ===
(1) same decision on 209 validation pairs: 100.0%
(2) true validation matches: 67
    removed by gate (title/author):   4
    removed by exclusion conditions:  0
    removed by step 3 (missing data): 0
    kept for 1:1:                     63

=== goodreads_dbpedia (without missing-data) ===
(1) same decision on 210 validation pairs: 100.0%
(2) true validation matches: 70
    removed by gate (title/author):   17
    removed by exclusion conditions:  5
    removed by step 3 (missing data): 0
    kept for 1:1:                     48


### 6.2 ML-based matchers
Grid search with 5-fold cross-validation on the training set, for each dataset pair
separately (the attributes available differ between the pairs). The model with the best
**validation** F1 is selected.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

MODELS = {
    "LogisticRegression": (LogisticRegression(max_iter=2000, random_state=SEED),
                           {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"]}),
    "RandomForest": (RandomForestClassifier(random_state=SEED),
                     {"n_estimators": [100, 300], "max_depth": [5, 10, None], "class_weight": [None, "balanced"]}),
    "GradientBoosting": (GradientBoostingClassifier(random_state=SEED),
                         {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]}),
}

model_rows, best_models = [], {}
for p in features:
    tr, va = features[p]["train"], features[p]["validation"]
    best_f1 = -1
    for name, (model, grid) in MODELS.items():
        gs = GridSearchCV(model, grid, scoring="f1", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
        gs.fit(tr[FEATURE_COLUMNS], tr["label"])
        val_scores = scores(va["label"], gs.best_estimator_.predict(va[FEATURE_COLUMNS]))
        model_rows.append({"pair": p, "model": name, "cv f1 (train)": gs.best_score_, **{f"val {k}": v for k, v in val_scores.items()},
                           "params": gs.best_params_})
        if val_scores["f1"] > best_f1:
            best_f1, best_models[p] = val_scores["f1"], (name, gs.best_estimator_)
    bl = scores(va["label"], baseline_score(va) >= baseline[p])
    model_rows.append({"pair": p, "model": "Rule baseline", "cv f1 (train)": None, **{f"val {k}": v for k, v in bl.items()},
                       "params": {"threshold": baseline[p]}})
model_results = pd.DataFrame(model_rows)
model_results.to_csv(OUT_DIR / "model_selection.csv", index=False)
display(model_results.style.format({c: "{:.3f}" for c in model_results.columns if "f1" in c or "precision" in c or "recall" in c}, na_rep="-"))
print({p: m[0] for p, m in best_models.items()})

,pair,model,cv f1 (train),val precision,val recall,val f1,params
0,goodreads_amazon,LogisticRegression,0.853,0.838,0.851,0.844,"{'C': 1.0, 'class_weight': None}"
1,goodreads_amazon,RandomForest,0.901,0.840,0.940,0.887,"{'class_weight': 'balanced', 'max_depth': 5, 'n_estimators': 100}"
2,goodreads_amazon,GradientBoosting,0.896,0.905,0.851,0.877,"{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 100}"
3,goodreads_amazon,Rule baseline,-,0.746,0.657,0.698,{'threshold': 0.99}
4,goodreads_dbpedia,LogisticRegression,0.733,0.683,0.800,0.737,"{'C': 10.0, 'class_weight': 'balanced'}"
5,goodreads_dbpedia,RandomForest,0.819,0.728,0.843,0.781,"{'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 100}"
6,goodreads_dbpedia,GradientBoosting,0.802,0.733,0.786,0.759,"{'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}"
7,goodreads_dbpedia,Rule baseline,-,0.466,0.886,0.611,{'threshold': 0.65}
8,amazon_dbpedia,LogisticRegression,0.726,0.700,0.860,0.772,"{'C': 10.0, 'class_weight': 'balanced'}"
9,amazon_dbpedia,RandomForest,0.832,0.717,0.754,0.735,"{'class_weight': 'balanced', 'max_depth': 10, 'n_estimators': 300}"


{'goodreads_amazon': 'RandomForest', 'goodreads_dbpedia': 'RandomForest', 'amazon_dbpedia': 'LogisticRegression'}


In [ ]:
# Feature importance of the selected models (tree models) / coefficients (logistic regression)
for p, (name, model) in best_models.items():
    imp = getattr(model, "feature_importances_", None)
    if imp is None:
        imp = abs(model.coef_[0])
    top = pd.Series(imp, index=FEATURE_COLUMNS).sort_values(ascending=False).head(6)
    print(f"\n{p} ({name})"); print(top.round(3).to_string())


goodreads_amazon (RandomForest)
NumericComparator(page_count, relative_difference, list_strategy=None)                 0.225
StringComparator(publisher_clean, jaccard, tokenization=word, list_strategy=None)      0.196
NumericComparator(year, absolute_difference, list_strategy=None)                       0.192
StringComparator(format_clean, jaro_winkler, tokenization=char, list_strategy=None)    0.130
StringComparator(title, jaccard, tokenization=word, list_strategy=None)                0.074
StringComparator(title_clean, jaro_winkler, tokenization=char, list_strategy=None)     0.045

goodreads_dbpedia (RandomForest)
NumericComparator(year, absolute_difference, list_strategy=None)                            0.213
NumericComparator(page_count, relative_difference, list_strategy=None)                      0.160
StringComparator(publisher_clean, jaccard, tokenization=word, list_strategy=None)           0.159
StringComparator(title_after_colon, jaro_winkler, tokenization=char, list_strateg

## 7. Apply to all candidates, one-to-one matching, test evaluation

The selected model is applied to all candidate pairs from blocking (score = match probability).

**Global (one-to-one) matching is chosen because of the data, not by validation F1.**
The lecture (WDI05, slide 19) recommends local matching for duplicate detection *within* one source
and global matching for sources that do *not* contain duplicates. Our sources are nearly
duplicate-free at edition level: only 0.2% of Goodreads, 2.8% of Amazon and 0.8% of DBpedia records
share an ISBN with another record of the same source. We therefore always apply greedy
one-to-one matching.

Only the **match threshold** (0.5 to 0.9) is tuned on the **validation** set (ties: stricter
threshold); the test set is used once for the final result. The variant without one-to-one is
reported only as a sanity check: its correspondence counts are far above the expected number of
true matches (see the plausibility check below), which confirms the choice. Matches lost in blocking
count as false negatives, so the results include the blocking loss.

In [ ]:
def chunks(df, size=100_000):
    for i in range(0, len(df), size):
        yield df.iloc[i:i + size]


results, correspondences = [], {}
for a, b in PAIRS:
    p = f"{a}_{b}"
    name, model = best_models[p]
    t = time.time()
    # use_probabilities=True: the score is the predicted match probability. Without it every
    # match gets score 1.0, and one-to-one matching would pick an arbitrary edition.
    corr = MLBasedMatcher(feature_extractor).match(data[a], data[b], candidates=list(chunks(candidates[p])),
                                                   id_column="id", trained_classifier=model,
                                                   use_probabilities=True, threshold=0.5)
    minutes = (time.time() - t) / 60

    # Each source lists every edition at most once, so we enforce one-to-one matching.
    # The match threshold is chosen on the VALIDATION set (ties -> stricter threshold);
    # the variant without one-to-one is only reported for comparison.
    def evaluate(c, split, tag):
        ev = EntityMatchingEvaluator.evaluate_matching(correspondences=c, test_pairs=gold[p][split],
                                                       out_dir=OUT_DIR / f"eval_{p}_{tag}_{split}")
        return ev["precision"], ev["recall"], ev["f1"]

    for split in ["validation", "test"]:
        P, R, F = evaluate(corr, split, "without_11")
        results.append({"pair": p, "model": name, "variant": "threshold 0.5, without 1:1", "split": split,
                        "correspondences": len(corr), "precision": P, "recall": R, "f1": F, "minutes": round(minutes, 1)})
    best = None
    for thr in [0.5, 0.6, 0.7, 0.8, 0.9]:
        c = GreedyOneToOneMatchingAlgorithm().cluster(corr[corr["score"] >= thr].reset_index(drop=True))
        P, R, F = evaluate(c, "validation", f"t{int(thr * 10)}_11")
        results.append({"pair": p, "model": name, "variant": f"threshold {thr}, with 1:1", "split": "validation",
                        "correspondences": len(c), "precision": P, "recall": R, "f1": F, "minutes": round(minutes, 1)})
        if best is None or F >= best[1]:
            best = (thr, F, c)
    thr, _, chosen = best
    P, R, F = evaluate(chosen, "test", f"t{int(thr * 10)}_11")
    results.append({"pair": p, "model": name, "variant": f"threshold {thr}, with 1:1 (selected)", "split": "test",
                    "correspondences": len(chosen), "precision": P, "recall": R, "f1": F, "minutes": round(minutes, 1)})
    correspondences[p] = chosen
    print(f"{p}: threshold {thr} chosen on validation -> {len(chosen):,} correspondences, test F1 {F:.3f}")
test_results = pd.DataFrame(results)
test_results.to_csv(OUT_DIR / "test_results.csv", index=False)
display(test_results.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "correspondences": "{:,}"}))

goodreads_amazon: threshold 0.5 chosen on validation -> 3,898 correspondences, test F1 0.874


goodreads_dbpedia: threshold 0.5 chosen on validation -> 8,058 correspondences, test F1 0.725


amazon_dbpedia: threshold 0.6 chosen on validation -> 3,684 correspondences, test F1 0.676


,pair,model,variant,split,correspondences,precision,recall,f1,minutes
0,goodreads_amazon,RandomForest,"threshold 0.5, without 1:1",validation,"13,023",0.840,0.940,0.887,8.700000
1,goodreads_amazon,RandomForest,"threshold 0.5, without 1:1",test,"13,023",0.806,0.897,0.849,8.700000
2,goodreads_amazon,RandomForest,"threshold 0.5, with 1:1",validation,"3,898",1.000,0.746,0.855,8.700000
3,goodreads_amazon,RandomForest,"threshold 0.6, with 1:1",validation,"3,193",1.000,0.731,0.845,8.700000
4,goodreads_amazon,RandomForest,"threshold 0.7, with 1:1",validation,"2,532",1.000,0.687,0.814,8.700000
5,goodreads_amazon,RandomForest,"threshold 0.8, with 1:1",validation,"2,172",1.000,0.642,0.782,8.700000
6,goodreads_amazon,RandomForest,"threshold 0.9, with 1:1",validation,"1,387",1.000,0.493,0.660,8.700000
7,goodreads_amazon,RandomForest,"threshold 0.5, with 1:1 (selected)",test,"3,898",0.987,0.784,0.874,8.700000
8,goodreads_dbpedia,RandomForest,"threshold 0.5, without 1:1",validation,"21,335",0.722,0.814,0.765,7.200000
9,goodreads_dbpedia,RandomForest,"threshold 0.5, without 1:1",test,"21,335",0.732,0.759,0.745,7.200000


In [ ]:
# ============================================================
# LINEAR RULE: TEST (T1) – compute and save, run ONCE
#    Weights frozen from the train cell, method + threshold frozen from validation.
#    Metrics are computed against the gold-standard TEST set.
# ============================================================
METHODS = ["grid 0-3", "grid 0-4", "grid 0-5", "least squares"]   # order = tie-break (simpler first)

# --- choice per pair, read from the saved validation results ---
val_cmp = pd.read_csv(OUT_DIR / "method_comparison_validation_1to1.csv")
best = (val_cmp.sort_values("threshold", ascending=False)
               .loc[lambda d: d.groupby(["pair", "method"])["f1"].idxmax()])     # ties -> stricter threshold
best["order"] = best["method"].map(METHODS.index)
final_choice = (best.sort_values(["pair", "f1", "order"], ascending=[True, False, True])
                    .groupby("pair").head(1).reset_index(drop=True))             # ties -> simpler method


def chunks(df, size=100_000):
    for i in range(0, len(df), size):
        yield df.iloc[i:i + size]


# --- same features as in validation (publisher_norm is added with add_pub_norm) ---
needed = ({col("StringComparator(title_clean, jaro_winkler"), col("StringComparator(title_after_colon, jaro_winkler")}
          | set(SIM_COL.values()) | set(PRESENT_COL.values())) - {PUB_NORM}
extractor = FeatureExtractor([c for c in comparators if c.name in needed])

# --- test, once per pair ---
test_rows = []
for a, b in PAIRS:
    p = f"{a}_{b}"
    choice = final_choice[final_choice["pair"] == p].iloc[0]
    method, thr = choice["method"], float(choice["threshold"])

    feats = pd.concat([extractor.create_features(data[a], data[b], c, id_column="id")
                       for c in chunks(candidates[p])], ignore_index=True)
    feats = add_pub_norm(feats, a, b)
    S_c, P_c = sims(feats)
    sc = feats[["id1", "id2"]].copy()
    sc["score"] = linear_score(S_c, P_c, weights[method][p])

    corr = GreedyOneToOneMatchingAlgorithm().cluster(sc[sc["score"] >= thr].reset_index(drop=True))
    ev = EntityMatchingEvaluator.evaluate_matching(correspondences=corr, test_pairs=gold[p]["test"])
    test_rows.append({"pair": p, "method": method, "threshold": thr,
                      "precision": ev["precision"], "recall": ev["recall"], "f1": ev["f1"],
                      "correspondences": len(corr)})
    corr.to_csv(CORR_DIR / f"{p}_rule.csv", index=False)      # separate file: ML correspondences are NOT overwritten
    print(p, "done")

# validation (chosen) + test in ONE file, with a "split" column (same format idea as Shiqi's test_results.csv)
val_part = final_choice[["pair", "method", "threshold", "precision", "recall", "f1", "correspondences"]].assign(split="validation")
test_part = pd.DataFrame(test_rows).assign(split="test")
results_linear_rule = (pd.concat([val_part, test_part], ignore_index=True)
                         .sort_values(["pair", "split"], ascending=[True, False]).reset_index(drop=True))
results_linear_rule.to_csv(OUT_DIR / "test_results_linear_rule.csv", index=False)
print("Saved: test_results_linear_rule.csv (validation + test)")

goodreads_amazon done
goodreads_dbpedia done
amazon_dbpedia done
Saved: test_results_linear_rule.csv (validation + test)


In [ ]:
# LINEAR RULE: chosen rule, validation + test (with 1:1), same format and order as Shiqi's test_results.csv
test_df = pd.DataFrame(test_rows)
val_part = final_choice.assign(split="validation")
test_part = test_df.assign(split="test")

results_linear_rule = pd.concat([val_part, test_part], ignore_index=True)
results_linear_rule["model"] = "LinearRule (" + results_linear_rule["method"] + ")"
results_linear_rule["variant"] = "threshold " + results_linear_rule["threshold"].astype(str) + ", with 1:1 (selected)"

# same pair order as Shiqi (PAIRS order), validation before test
pair_order = [f"{a}_{b}" for a, b in PAIRS]
results_linear_rule["pair"] = pd.Categorical(results_linear_rule["pair"], categories=pair_order, ordered=True)
results_linear_rule["split"] = pd.Categorical(results_linear_rule["split"], categories=["validation", "test"], ordered=True)
results_linear_rule = (results_linear_rule[["pair", "model", "variant", "split", "correspondences", "precision", "recall", "f1"]]
                       .sort_values(["pair", "split"]).reset_index(drop=True))

results_linear_rule.to_csv(OUT_DIR / "test_results_linear_rule.csv", index=False)
display(results_linear_rule.round(3))

,pair,model,variant,split,correspondences,precision,recall,f1
0,goodreads_amazon,LinearRule (grid 0-3),"threshold 0.8, with 1:1 (selected)",validation,5107,0.979,0.687,0.807
1,goodreads_amazon,LinearRule (grid 0-3),"threshold 0.8, with 1:1 (selected)",test,5107,0.937,0.608,0.738
2,goodreads_dbpedia,LinearRule (grid 0-5),"threshold 0.7, with 1:1 (selected)",validation,11572,0.679,0.814,0.740
3,goodreads_dbpedia,LinearRule (grid 0-5),"threshold 0.7, with 1:1 (selected)",test,11572,0.637,0.796,0.708
4,amazon_dbpedia,LinearRule (grid 0-4),"threshold 0.5, with 1:1 (selected)",validation,9645,0.705,0.544,0.614
5,amazon_dbpedia,LinearRule (grid 0-4),"threshold 0.5, with 1:1 (selected)",test,9645,0.741,0.449,0.559


In [ ]:
# ============================================================
# NON-LINEAR RULE: TEST (D1) – evaluated ONCE with the choice from cell C.
# No changes to the rule after this cell.
# ============================================================
nl_test_rows = []
for _, r in nl_final.iterrows():
    p, feats = r["pair"], cand_feats[r["pair"]]
    S_c, P_c = sims(feats)
    keep = nonlinear_pred(S_c, P_c, nl_th[p, r["variant"]])
    sc = pd.DataFrame({"id1": feats["id1"], "id2": feats["id2"],
                       "score": linear_score(S_c, P_c, weights[CHOSEN[p]][p])})[keep]
    c = GreedyOneToOneMatchingAlgorithm().cluster(sc[sc["score"] >= r["threshold"]].reset_index(drop=True))
    ev = EntityMatchingEvaluator.evaluate_matching(correspondences=c, test_pairs=gold[p]["test"])
    nl_test_rows.append({"pair": p, "variant": r["variant"], "threshold": r["threshold"],
                         "correspondences": len(c), "precision": ev["precision"], "recall": ev["recall"], "f1": ev["f1"]})

In [ ]:
# NON-LINEAR RULE: chosen rule, validation + test (with 1:1), same format and order as Shiqi's test_results.csv
nl_val_part = nl_final[["pair", "variant", "threshold", "correspondences", "precision", "recall", "f1"]].assign(split="validation")
nl_test_part = pd.DataFrame(nl_test_rows).assign(split="test")

results_nonlinear_rule = pd.concat([nl_val_part, nl_test_part], ignore_index=True)
results_nonlinear_rule["model"] = "NonLinearRule (" + results_nonlinear_rule["variant"] + ")"
results_nonlinear_rule["variant"] = "threshold " + results_nonlinear_rule["threshold"].astype(str) + ", with 1:1 (selected)"

# same pair order as Shiqi (PAIRS order), validation before test
pair_order = [f"{a}_{b}" for a, b in PAIRS]
results_nonlinear_rule["pair"] = pd.Categorical(results_nonlinear_rule["pair"], categories=pair_order, ordered=True)
results_nonlinear_rule["split"] = pd.Categorical(results_nonlinear_rule["split"], categories=["validation", "test"], ordered=True)
results_nonlinear_rule = (results_nonlinear_rule[["pair", "model", "variant", "split", "correspondences", "precision", "recall", "f1"]]
                          .sort_values(["pair", "split"]).reset_index(drop=True))

results_nonlinear_rule.to_csv(OUT_DIR / "test_results_nonlinear_rule.csv", index=False)
display(results_nonlinear_rule.round(3))

,pair,model,variant,split,correspondences,precision,recall,f1
0,goodreads_amazon,NonLinearRule (gate only),"threshold 0.7, with 1:1 (selected)",validation,3414,0.958,0.687,0.800
1,goodreads_amazon,NonLinearRule (gate only),"threshold 0.7, with 1:1 (selected)",test,3414,0.940,0.649,0.768
2,goodreads_dbpedia,NonLinearRule (without missing-data),"threshold 0.7, with 1:1 (selected)",validation,5545,0.640,0.686,0.662
3,goodreads_dbpedia,NonLinearRule (without missing-data),"threshold 0.7, with 1:1 (selected)",test,5545,0.650,0.704,0.676
4,amazon_dbpedia,NonLinearRule (gate only),"threshold 0.0, with 1:1 (selected)",validation,2703,0.711,0.474,0.568
5,amazon_dbpedia,NonLinearRule (gate only),"threshold 0.0, with 1:1 (selected)",test,2703,0.769,0.449,0.567


### Plausibility check
The gold standard only contains ~1,200 pairs per dataset pair, so wrong correspondences *outside* it do not lower the scores (lecture: "your evaluation is only as good as your evaluation set"). As a sanity check we compare the number of correspondences with a rough estimate of the true overlap: shared ISBNs divided by the share of records that have an ISBN on both sides.

In [ ]:
from overlap_estimate import load_goodreads, load_dbpedia, load_amazon, isbn_set

isbn_data = {"goodreads": load_goodreads(), "amazon": load_amazon("data/amazon_books_subset.csv"), "dbpedia": load_dbpedia()}
rows = []
for a, b in PAIRS:
    A, B = isbn_data[a], isbn_data[b]
    shared = len(isbn_set(A) & isbn_set(B))
    cover = (A.isbns.str.len() > 0).mean() * (B.isbns.str.len() > 0).mean()
    rows.append({"pair": f"{a}_{b}", "shared ISBNs": shared, "estimated true matches": round(shared / cover),
                 "correspondences without 1:1": int(test_results[(test_results.pair == f"{a}_{b}") & test_results.variant.str.contains("without")].correspondences.iloc[0]),
                 "final correspondences": len(correspondences[f"{a}_{b}"])})
display(pd.DataFrame(rows))

,pair,shared ISBNs,estimated true matches,correspondences without 1:1,final correspondences
0,goodreads_amazon,2169,2642,13023,3898
1,goodreads_dbpedia,3271,6590,21335,8058
2,amazon_dbpedia,1725,2854,40698,3684


### Cluster size distribution
With one-to-one matching inside each pair, larger clusters can only come from combining the
three pairs. Many clusters larger than 3 would point to wrong correspondences.

In [ ]:
all_corr = pd.concat(correspondences.values(), ignore_index=True)
dist = EntityMatchingEvaluator.create_cluster_size_distribution(correspondences=all_corr, out_dir=OUT_DIR / "cluster_analysis")
display(dist)

,cluster_size,frequency,percentage
0,2,10209,83.454590
1,3,1359,11.109295
2,4,536,4.381591
3,5,62,0.506826
4,6,35,0.286111
5,7,21,0.171667
6,8,5,0.040873
7,9,3,0.024524
8,10,3,0.024524


### Cluster cleanup across pairs
One-to-one matching holds inside each pair, but combining the three pairs can still create
clusters with two records of the same source (665 of 12,233 clusters, about 5%). Since each source
lists an edition at most once, such a cluster contains at least one wrong correspondence.
We go through all correspondences from highest to lowest score and keep one only if it does not
put two records of the same source into one cluster (`cluster_cleanup.py`). Afterwards every
cluster has at most one record per source (size 2 or 3).

Effect on the test set: precision stays the same (the gold standard contains few of the removed
pairs), recall drops by 0 to 3 points, because in a conflict the higher-scored correspondence is not
always the correct one. We accept this so that data fusion never merges two editions of the same source.
Scores come from different models per pair and are therefore only roughly comparable.

In [ ]:
from cluster_cleanup import enforce_one_per_source, test_scores

before = correspondences
correspondences = enforce_one_per_source(correspondences)
rows = []
for p in correspondences:
    pb, rb, fb = test_scores(before[p], p)
    pa, ra, fa = test_scores(correspondences[p], p)
    rows.append({"pair": p, "before": len(before[p]), "after": len(correspondences[p]),
                 "test P": f"{pb:.3f} -> {pa:.3f}", "test R": f"{rb:.3f} -> {ra:.3f}", "test F1": f"{fb:.3f} -> {fa:.3f}"})
display(pd.DataFrame(rows))

all_corr = pd.concat(correspondences.values(), ignore_index=True)
display(EntityMatchingEvaluator.create_cluster_size_distribution(correspondences=all_corr, out_dir=OUT_DIR / "cluster_analysis_cleaned"))

## 8. Error analysis (test set)
False positives and false negatives of the final correspondences, with the main attributes.

In [ ]:
def error_table(p, n=8):
    a, b = p.split("_")
    A, B = data[a].set_index("id"), data[b].set_index("id")
    pred = set(zip(correspondences[p].id1, correspondences[p].id2))
    te = gold[p]["test"]
    te = te.assign(pred=[(x, y) in pred for x, y in zip(te.id1, te.id2)])
    cols = ["title", "first_author", "publisher", "year", "page_count", "format"]
    out = []
    for kind, mask in [("false positive", te.pred & ~te.label), ("false negative", ~te.pred & te.label)]:
        for x, y in zip(te.loc[mask, "id1"].head(n), te.loc[mask, "id2"].head(n)):
            out.append({"error": kind, **{f"{c} 1": A.at[x, c] for c in cols}, **{f"{c} 2": B.at[y, c] for c in cols}})
    return pd.DataFrame(out)


for p in correspondences:
    print(p)
    display(error_table(p))

goodreads_amazon


,error,title 1,first_author 1,publisher 1,year 1,page_count 1,format 1,title 2,first_author 2,publisher 2,year 2,page_count 2,format 2
0,false positive,Two for the Dough,Janet Evanovich,Penguin Books,<NA>,326.0,Paperback,"Two for the Dough (Stephanie Plum, No. 2) (Ste...",Janet Evanovich,Pocket Books,1996,336.0,Paperback
1,false negative,Stormbreaker,Anthony Horowitz,Speak,2004,256.0,Paperback,Stormbreaker (Alex Rider),Anthony Horowitz,Penguin,2004,256.0,Paperback
2,false negative,Maybe in Another Life,Taylor Jenkins Reid,"Dreamscape Media, LLC",2015,10.0,Audiobook,Maybe in Another Life: A Novel,Taylor Jenkins Reid,Washington Square Press,2015,352.0,Paperback
3,false negative,Rich People Problems,Kevin Kwan,Doubleday,2017,398.0,Hardcover,Rich People Problems (Crazy Rich Asians Trilogy),Kevin Kwan,Doubleday,2017,416.0,Hardcover
4,false negative,The Stranger in the Woods: The Extraordinary S...,Michael Finkel,Books on Tape,2017,224.0,Audio CD,The Stranger in the Woods,Michael Finkel,BooksOnTape,2017,NaN,Audio CD
5,false negative,"Oh, the Thinks You Can Think!",Dr. Seuss,Beginner Books,1975,64.0,Paperback,"Oh, the Thinks You Can Think! Green Back Book",Dr. Seuss,Harpercollins Childs,2004,64.0,Paperback
6,false negative,Now You See Her,James Patterson,"Little, Brown and Company",2011,359.0,Mass Market Paperback,Now You See Her,James Patterson,"Little, Brown and Company",2011,432.0,Hardcover
7,false negative,It,Stephen King,New English Library,<NA>,1116.0,Paperback,It,Stephen King,Trafalgar Square,1987,920.0,Paperback
8,false negative,Chicka Chicka Boom Boom,Bill Martin Jr.,Beach Lane Books,2000,40.0,Paperback,Chicka Chicka Boom Boom,John Archambault,Beach Lane Books,2000,32.0,Paperback


goodreads_dbpedia


,error,title 1,first_author 1,publisher 1,year 1,page_count 1,format 1,title 2,first_author 2,publisher 2,year 2,page_count 2,format 2
0,false positive,Olvidado rey Gudú,Ana María Matute,Espasa Calpe,1996,872.0,Hardcover,Olvidado rey Gudú,Ana María Matute,Espasa-Calpe,1996,865.0,None
1,false positive,Reservation Blues,Sherman Alexie,Grove Press,2005,306.0,Paperback,Reservation Blues,Sherman Alexie,NaN,1995,306.0,None
2,false positive,Oscar and Lucinda,Peter Carey,University of Queensland Press,1998,515.0,Paperback,Oscar and Lucinda,Peter Carey,University of Queensland Press,1988,528.0,None
3,false positive,The Prophecy,Chris Kuzneski,Penguin,2009,547.0,Paperback,The Prophecy (Kuzneski novel),Chris Kuzneski,G. P. Putnam's Sons,2009,NaN,None
4,false positive,Death of Kings,Bernard Cornwell,HarperCollins Publishers,2011,335.0,Hardcover,Death of Kings,Bernard Cornwell,HarperCollins,2011,335.0,None
5,false positive,Tietam Brown,Mick Foley,Knopf,2003,256.0,Hardcover,Tietam Brown,Mick Foley,Alfred A. Knopf,2003,243.0,None
6,false positive,Sword of God,Chris Kuzneski,Penguin,2007,432.0,Paperback,Sword of God (novel),Chris Kuzneski,Penguin Books,2007,NaN,None
7,false positive,Drift: The Unmooring of American Military Power,Rachel Maddow,Random House Audio,2012,288.0,NaN,Drift: The Unmooring of American Military Power,Rachel Maddow,Crown Publishing Group,2012,288.0,None
8,false negative,Enrique's Journey,Sonia Nazario,Random House Trade Paperbacks,2007,294.0,Paperback,Enrique's Journey,,skid,<NA>,NaN,None
9,false negative,Timeless,Alexandra Monir,Delacorte Press,2011,280.0,Hardcover,Timeless (Monir novel),Alexandra Monir,Random House Children's Books,<NA>,304.0,None


amazon_dbpedia


,error,title 1,first_author 1,publisher 1,year 1,page_count 1,format 1,title 2,first_author 2,publisher 2,year 2,page_count 2,format 2
0,false positive,"Even Dogs in the Wild (A Rebus Novel, 20)",Ian Rankin,"Little, Brown and Company",2016,352.0,Hardcover,Even Dogs in the Wild,Ian Rankin,Orion Publishing Group,2015,352.0,None
1,false positive,Life,James Fox,"Little, Brown and Company",2010,576.0,Hardcover,Life (Richards book),James Fox,"Little, Brown and Company",2010,564.0,None
2,false positive,Scythe (1) (Arc of a Scythe),Neal Shusterman,Simon & Schuster Books for Young Readers,2016,448.0,Hardcover,Scythe (novel),Neal Shusterman,Simon & Schuster,2016,443.0,None
3,false positive,"Where'd You Go, Bernadette: A Novel",Maria Semple,"Little, Brown and Company",2012,336.0,Hardcover,"Where'd You Go, Bernadette",Maria Semple,"Little, Brown and Company",2012,336.0,None
4,false positive,The Wind-Up Bird Chronicle,Haruki Murakami,Knopf,1997,611.0,Hardcover,The Wind-Up Bird Chronicle,Haruki Murakami,NaN,1997,607.0,None
5,false positive,"The Happiness Project (Revised Edition): Or, W...",Gretchen Rubin,Harper Paperbacks,2015,368.0,Paperback,"Nazis, Islamists, and the Making of the Modern...",Barry Rubin,Yale University Press,2014,360.0,None
6,false positive,The Night Manager,John le Carré,Ballantine Books,1994,443.0,Paperback,The Night Manager,John le Carré,Hodder & Stoughton,1993,443.0,None
7,false positive,Team of Rivals,Doris Kearns Goodwin,Simon & Schuster,2005,944.0,Hardcover,Team of Rivals,Doris Kearns Goodwin,Simon & Schuster,2005,944.0,None
8,false negative,Warriors: Into the Wild,Erin Hunter,HarperCollins,2003,288.0,Hardcover,Into the Wild (novel),Erin Hunter,Avon (publisher),2003,272.0,None
9,false negative,A Land Remembered,Patrick D. Smith,Pineapple Press,1998,416.0,Hardcover,A Land Remembered,Patrick D. Smith,Pineapple Press,1984,403.0,None


## 9. Save correspondences for data fusion
One CSV per dataset pair (`id1, id2, score`) in `data/correspondences/`.

In [ ]:
for p, c in correspondences.items():
    c.to_csv(CORR_DIR / f"{p}.csv", index=False)
    print(f"{p:<20} {len(c):>6,} correspondences -> {CORR_DIR / (p + '.csv')}")

goodreads_amazon      3,898 correspondences -> data/correspondences/goodreads_amazon.csv
goodreads_dbpedia     8,058 correspondences -> data/correspondences/goodreads_dbpedia.csv
amazon_dbpedia        3,684 correspondences -> data/correspondences/amazon_dbpedia.csv
